<a href="https://colab.research.google.com/github/Miranita-ar/Skripsi-Gojek-App-Review/blob/main/Code/(XC)_Skripsi_IndoBERT_(SHAP_Analysis).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ============================================================
# EXPLAINABLE ARTIFICIAL INTELLIGENCE (XAI)
# Analisis Sentimen Ulasan Aplikasi Gojek
# Menggunakan Model IndoBERT Base P2
# ============================================================

## Peneliti

**Miranita Anisa Rohmah**

NIM : **11220940000055**

Program Studi : **Matematika**

Fakultas : **Fakultas Sains dan Teknologi**

Universitas : **UIN Syarif Hidayatullah Jakarta**

---

## Notebook

**XC — SHAP Analysis**

---

## Tujuan Notebook

Notebook ini bertujuan menjelaskan keputusan prediksi model
IndoBERT menggunakan metode **SHAP (SHapley Additive
exPlanations)** terhadap **30 sampel hasil purposive sampling**
yang telah dipilih pada Notebook XB.

Analisis dilakukan pada tiga kelompok utama:

- High Confidence
- Low Confidence
- Error Analysis

serta dilengkapi dengan analisis SHAP keseluruhan
(Overall SHAP Analysis).

---

## Versi

**1.0**

Status :

**Development**

Update terakhir :

**27 Juni 2026**

---

## Perubahan

### v1.0

- Struktur notebook dibuat
- Implementasi SHAP Local Analysis
- Implementasi SHAP Group Analysis
- Implementasi SHAP Overall Analysis
- Penyimpanan output otomatis
- Validasi otomatis
- Dokumentasi otomatis

---

BAB 1
Informasi Proyek

BAB 2
Persiapan Lingkungan
- Install library
- Import library
- Mount Drive
- Konfigurasi path

BAB 3
Load Model IndoBERT

BAB 4
Load Dataset Hasil Notebook XB

BAB 5
Membangun SHAP Explainer

BAB 6
Menjalankan SHAP

BAB 7
Visualisasi SHAP
- Text Plot
- Waterfall Plot
- Bar Plot
- Beeswarm Plot
- Summary Plot

BAB 8
Menyimpan Seluruh Output SHAP

BAB 9
Ringkasan Notebook

# 1. Persiapan

## 1.1 Install Library

In [ ]:
# =====================================================
# CELL 1 : INSTALL LIBRARY
# =====================================================

!pip install -q transformers
!pip install -q datasets
!pip install -q accelerate
!pip install -q sentencepiece
!pip install -q shap

print("=" * 60)
print("INSTALL LIBRARY")
print("=" * 60)

print()

print("✅ Seluruh library berhasil diinstal.")

INSTALL LIBRARY

✅ Seluruh library berhasil diinstal.


## 1.2 Import Library

In [ ]:
# =====================================================
# CELL 2 : IMPORT LIBRARY
# =====================================================

# Library umum
import os
import gc
import json
import pickle
import random
import warnings

# Data
import numpy as np
import pandas as pd

# Visualisasi
import matplotlib.pyplot as plt

# Deep Learning
import torch
import torch.nn.functional as F

# HuggingFace
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# Explainable AI
import shap

# Progress Bar
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("=" * 60)
print("IMPORT LIBRARY")
print("=" * 60)

print()

print("✅ Seluruh library berhasil diimpor.")

IMPORT LIBRARY

✅ Seluruh library berhasil diimpor.


## 1.3 Mount Google Drive

In [ ]:
# =====================================================
# CELL 3 : MOUNT GOOGLE DRIVE
# =====================================================

from google.colab import drive

drive.mount("/content/drive")

print("=" * 60)
print("MOUNT GOOGLE DRIVE")
print("=" * 60)

print()

print("✅ Google Drive berhasil dihubungkan.")

Mounted at /content/drive
MOUNT GOOGLE DRIVE

✅ Google Drive berhasil dihubungkan.


## 1.4 Konfigurasi Fungsi

In [ ]:
# =====================================================
# CELL 4 : KONFIGURASI FUNGSI
# =====================================================

def print_header(title):
    """
    Menampilkan judul proses.
    """
    print("=" * 60)
    print(title)
    print("=" * 60)


def print_success(message):
    """
    Menampilkan pesan sukses.
    """
    print(f"✅ {message}")


def print_info(message):
    """
    Menampilkan informasi.
    """
    print(f"ℹ️ {message}")


def print_warning(message):
    """
    Menampilkan peringatan.
    """
    print(f"⚠️ {message}")


def print_error(message):
    """
    Menampilkan pesan error.
    """
    print(f"❌ {message}")


print_header("KONFIGURASI FUNGSI")

print_success("Seluruh helper function berhasil dibuat.")

KONFIGURASI FUNGSI
✅ Seluruh helper function berhasil dibuat.


## 1.5 Konfigurasi Path

In [ ]:
# =====================================================
# CELL 5 : KONFIGURASI PATH
# =====================================================

PATHS = {

    # Root Project
    "PROJECT":
        "/content/drive/MyDrive/Skripsi_IndoBERT",

    # Model Final (Read Only)
    "MODEL":
        "/content/drive/MyDrive/Skripsi_IndoBERT/model_indobert_terbaik",

    # Notebook XB
    "XB":
        "/content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI",

    "XB_CSV":
        "/content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/csv",

    "XB_LOG":
        "/content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/log",

    # Notebook XC
    "XC":
        "/content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis"

}

print_header("KONFIGURASI PATH")

print()

for key, value in PATHS.items():

    print_info(f"{key:<8}: {value}")

print()

print_success("Seluruh path berhasil dikonfigurasi.")

KONFIGURASI PATH

ℹ️ PROJECT : /content/drive/MyDrive/Skripsi_IndoBERT
ℹ️ MODEL   : /content/drive/MyDrive/Skripsi_IndoBERT/model_indobert_terbaik
ℹ️ XB      : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI
ℹ️ XB_CSV  : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/csv
ℹ️ XB_LOG  : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/log
ℹ️ XC      : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis

✅ Seluruh path berhasil dikonfigurasi.


## 1.6 Konfigurasi Folder XC

In [ ]:
# =====================================================
# CELL 6 : KONFIGURASI FOLDER XC
# =====================================================

XC_DIR = PATHS["XC"]

LOCAL_DIR = os.path.join(XC_DIR, "local")

GROUP_DIR = os.path.join(XC_DIR, "group")

OVERALL_DIR = os.path.join(XC_DIR, "overall")

TABLE_DIR = os.path.join(XC_DIR, "tables")

METADATA_DIR = os.path.join(XC_DIR, "metadata")

README_DIR = os.path.join(XC_DIR, "readme")

folders = [

    XC_DIR,

    LOCAL_DIR,

    GROUP_DIR,

    OVERALL_DIR,

    TABLE_DIR,

    METADATA_DIR,

    README_DIR

]

for folder in folders:

    os.makedirs(folder, exist_ok=True)

print_header("KONFIGURASI FOLDER XC")

print()

for folder in folders:

    print(f"📁 {folder}")

print()

print_success("Seluruh folder Notebook XC berhasil dibuat.")

KONFIGURASI FOLDER XC

📁 /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis
📁 /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/local
📁 /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/group
📁 /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/overall
📁 /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/tables
📁 /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/metadata
📁 /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/readme

✅ Seluruh folder Notebook XC berhasil dibuat.


# 2. Load Model

## 2.1 Load Model IndoBERT

In [ ]:
# =====================================================
# CELL 7 : LOAD MODEL INDOBERT
# =====================================================

MODEL_DIR = PATHS["MODEL"]

print_header("LOAD MODEL INDOBERT")

print()

print_info(f"Model Directory : {MODEL_DIR}")

if not os.path.exists(MODEL_DIR):

    raise FileNotFoundError(

        f"Folder model tidak ditemukan:\n{MODEL_DIR}"

    )

model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_DIR

)

print()

print_success("Model IndoBERT berhasil dimuat.")

LOAD MODEL INDOBERT

ℹ️ Model Directory : /content/drive/MyDrive/Skripsi_IndoBERT/model_indobert_terbaik


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]


✅ Model IndoBERT berhasil dimuat.


## 2.2 Load Tokenizer

In [ ]:
# =====================================================
# CELL 8 : LOAD TOKENIZER
# =====================================================

print_header("LOAD TOKENIZER")

print()

tokenizer = AutoTokenizer.from_pretrained(

    MODEL_DIR

)

print_success("Tokenizer berhasil dimuat.")

LOAD TOKENIZER

✅ Tokenizer berhasil dimuat.


## 2.3 Validasi Model

In [ ]:
# =====================================================
# CELL 9 : VALIDASI MODEL
# =====================================================

DEVICE = torch.device(

    "cuda" if torch.cuda.is_available()

    else "cpu"

)

model.to(DEVICE)

model.eval()

label_mapping = {

    0: "Negatif",

    1: "Netral",

    2: "Positif"

}

print_header("VALIDASI MODEL")

print()

print_info(f"Device       : {DEVICE}")

print_info(f"Model Type   : {model.config.model_type}")

print_info(f"Jumlah Label : {model.config.num_labels}")

print()

print("Mapping Label")

for key, value in label_mapping.items():

    print(f"{key} -> {value}")

print()

print_success("Model siap digunakan untuk analisis SHAP.")

VALIDASI MODEL

ℹ️ Device       : cuda
ℹ️ Model Type   : bert
ℹ️ Jumlah Label : 3

Mapping Label
0 -> Negatif
1 -> Netral
2 -> Positif

✅ Model siap digunakan untuk analisis SHAP.


# 3. Load Dataset

## 3.1 Load Dataset Hasil Purposive Sampling

In [ ]:
# =====================================================
# CELL 10 : LOAD DATASET
# =====================================================

SELECTED_DATA_PATH = os.path.join(
    PATHS["XB_CSV"],
    "selected_reviews_xai.csv"
)

print_header("LOAD DATASET")

print()

print_info(f"Dataset : {SELECTED_DATA_PATH}")

if not os.path.exists(SELECTED_DATA_PATH):

    raise FileNotFoundError(
        f"Dataset tidak ditemukan:\n{SELECTED_DATA_PATH}"
    )

selected_reviews_df = pd.read_csv(
    SELECTED_DATA_PATH
)

print()

print_success("Dataset berhasil dimuat.")

print_info(
    f"Jumlah Sampel : {len(selected_reviews_df)}"
)

LOAD DATASET

ℹ️ Dataset : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/csv/selected_reviews_xai.csv

✅ Dataset berhasil dimuat.
ℹ️ Jumlah Sampel : 30


## 3.2 Validasi Dataset

In [ ]:
# =====================================================
# CELL 11 : VALIDASI DATASET
# =====================================================

print_header("VALIDASI DATASET")

print()

display(
    selected_reviews_df.head()
)

print()

print_info(
    f"Jumlah Baris : {len(selected_reviews_df)}"
)

print_info(
    f"Jumlah Kolom : {selected_reviews_df.shape[1]}"
)

# Kolom wajib
required_columns = [

    "sample_id",

    "text",

    "actual_label",

    "predicted_label",

    "prediction_case",

    "sample_group"

]

missing_columns = [

    col

    for col in required_columns

    if col not in selected_reviews_df.columns

]

duplicate_count = selected_reviews_df.duplicated(
    subset="text"
).sum()

missing_value = selected_reviews_df.isnull().sum().sum()

print_info(
    f"Jumlah Duplikat : {duplicate_count}"
)

print_info(
    f"Missing Value : {missing_value}"
)

print()

if len(missing_columns) == 0:

    print_success(
        "Seluruh kolom penting tersedia."
    )

else:

    print_warning(
        f"Kolom hilang : {missing_columns}"
    )

if duplicate_count == 0:

    print_success(
        "Tidak ditemukan text duplikat."
    )

else:

    print_warning(
        "Masih terdapat text duplikat."
    )

if missing_value == 0:

    print_success(
        "Tidak ditemukan missing value."
    )

else:

    print_warning(
        "Masih terdapat missing value."
    )

VALIDASI DATASET



,analysis_order,sample_id,text,actual_label,actual_sentiment,predicted_label,predicted_sentiment,prob_negatif,prob_netral,prob_positif,confidence,correct,prediction_case,confidence_level,selection_reason,sample_group,explain_status
0,1,XAI_001,sangat membantu dan mudah,2,Positif,2,Positif,0.000885,0.000943,0.998172,0.998172,True,Benar_Positif,Tinggi,Prediksi benar kelas Positif,High Confidence,Pending
1,2,XAI_002,mudah dan sangat membantu sekali,2,Positif,2,Positif,0.000879,0.000953,0.998167,0.998167,True,Benar_Positif,Tinggi,Prediksi benar kelas Positif,High Confidence,Pending
2,3,XAI_003,sangat membantu sekali,2,Positif,2,Positif,0.000898,0.000940,0.998162,0.998162,True,Benar_Positif,Tinggi,Prediksi benar kelas Positif,High Confidence,Pending
3,4,XAI_004,baik dan sangat membantu,2,Positif,2,Positif,0.000917,0.000924,0.998159,0.998159,True,Benar_Positif,Tinggi,Prediksi benar kelas Positif,High Confidence,Pending
4,5,XAI_005,sangat membantu sekali dalam segala aktivitas,2,Positif,2,Positif,0.000891,0.000954,0.998155,0.998155,True,Benar_Positif,Tinggi,Prediksi benar kelas Positif,High Confidence,Pending



ℹ️ Jumlah Baris : 30
ℹ️ Jumlah Kolom : 17
ℹ️ Jumlah Duplikat : 0
ℹ️ Missing Value : 0

✅ Seluruh kolom penting tersedia.
✅ Tidak ditemukan text duplikat.
✅ Tidak ditemukan missing value.


## 3.3 Load Sampling Log

In [ ]:
# =====================================================
# CELL 12 : LOAD SAMPLING LOG
# =====================================================

SAMPLING_LOG_PATH = os.path.join(
    PATHS["XB_LOG"],
    "sampling_log.csv"
)

print_header("LOAD SAMPLING LOG")

print()

print_info(f"File : {SAMPLING_LOG_PATH}")

if not os.path.exists(SAMPLING_LOG_PATH):

    raise FileNotFoundError(
        f"File tidak ditemukan:\n{SAMPLING_LOG_PATH}"
    )

sampling_log_df = pd.read_csv(
    SAMPLING_LOG_PATH
)

print()

display(
    sampling_log_df
)

print()

print_success(
    "Sampling log berhasil dimuat."
)

LOAD SAMPLING LOG

ℹ️ File : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/log/sampling_log.csv



,prediction_case,target,available,selected
0,Benar_Positif,5,752,5
1,Benar_Negatif,5,551,5
2,Benar_Netral,5,8,5
3,Salah_Netral_ke_Positif,5,16,5
4,Salah_Netral_ke_Negatif,5,51,5



✅ Sampling log berhasil dimuat.


## 3.4 Validasi Distribusi Sampel

In [ ]:
# =====================================================
# CELL 13 : VALIDASI DISTRIBUSI SAMPEL
# =====================================================

print_header("VALIDASI DISTRIBUSI SAMPEL")

print()

print("Distribusi Sample Group")

display(
    selected_reviews_df["sample_group"].value_counts()
)

print()

print("Distribusi Prediction Case")

display(
    selected_reviews_df["prediction_case"].value_counts()
)

print()

print("Cross Tab")

cross_tab = pd.crosstab(
    selected_reviews_df["sample_group"],
    selected_reviews_df["prediction_case"]
)

display(cross_tab)

print()

expected_total = 30
actual_total = len(selected_reviews_df)

print_info(f"Jumlah Sampel : {actual_total}")

if actual_total == expected_total:

    print_success(
        "Jumlah sampel sesuai dengan hasil purposive sampling."
    )

else:

    print_warning(
        f"Jumlah sampel tidak sesuai ({actual_total}/30)."
    )

print()

# Validasi setiap prediction_case
expected_prediction = {

    "Benar_Positif": 5,
    "Benar_Negatif": 5,
    "Benar_Netral": 5,
    "Salah_Netral_ke_Positif": 5,
    "Salah_Netral_ke_Negatif": 5,
    "Benar_Confidence_Rendah": 5

}

prediction_count = (
    selected_reviews_df["prediction_case"]
    .value_counts()
    .to_dict()
)

valid = True

for key, target in expected_prediction.items():

    current = prediction_count.get(key, 0)

    if current != target:

        valid = False

        print_warning(
            f"{key} = {current} (seharusnya {target})"
        )

if valid:

    print_success(
        "Distribusi prediction_case sesuai."
    )

print()

expected_group = {

    "High Confidence": 15,
    "Error Analysis": 10,
    "Low Confidence": 5

}

group_count = (
    selected_reviews_df["sample_group"]
    .value_counts()
    .to_dict()
)

valid_group = True

for key, target in expected_group.items():

    current = group_count.get(key, 0)

    if current != target:

        valid_group = False

        print_warning(
            f"{key} = {current} (seharusnya {target})"
        )

if valid_group:

    print_success(
        "Distribusi sample_group sesuai."
    )

VALIDASI DISTRIBUSI SAMPEL

Distribusi Sample Group


,count
sample_group,
High Confidence,15
Error Analysis,10
Low Confidence,5



Distribusi Prediction Case


,count
prediction_case,
Benar_Positif,5
Benar_Negatif,5
Benar_Netral,5
Salah_Netral_ke_Positif,5
Salah_Netral_ke_Negatif,5
Benar_Confidence_Rendah,5



Cross Tab


prediction_case,Benar_Confidence_Rendah,Benar_Negatif,Benar_Netral,Benar_Positif,Salah_Netral_ke_Negatif,Salah_Netral_ke_Positif
sample_group,,,,,,
Error Analysis,0,0,0,0,5,5
High Confidence,0,5,5,5,0,0
Low Confidence,5,0,0,0,0,0



ℹ️ Jumlah Sampel : 30
✅ Jumlah sampel sesuai dengan hasil purposive sampling.

✅ Distribusi prediction_case sesuai.

✅ Distribusi sample_group sesuai.


# 4. Persiapan Analisis SHAP

## 4.1 Konfigurasi Label

In [ ]:
# =====================================================
# CELL 14 : KONFIGURASI LABEL
# =====================================================

LABEL_NAMES = {

    0: "Negatif",

    1: "Netral",

    2: "Positif"

}

LABEL_COLORS = {

    "Negatif": "#d62728",

    "Netral": "#ffbf00",

    "Positif": "#2ca02c"

}

print_header("KONFIGURASI LABEL")

print()

for key, value in LABEL_NAMES.items():

    print_info(f"{key} -> {value}")

print()

print_success(
    "Konfigurasi label berhasil dibuat."
)

KONFIGURASI LABEL

ℹ️ 0 -> Negatif
ℹ️ 1 -> Netral
ℹ️ 2 -> Positif

✅ Konfigurasi label berhasil dibuat.


## 4.2 Konfigurasi SHAP

In [ ]:
# =====================================================
# CELL 15 : KONFIGURASI SHAP
# =====================================================

MODEL_CONFIG = {

    # Mengikuti konfigurasi fine-tuning terbaik
    "max_length": 128,

    # Batch inferensi SHAP
    "batch_size": 8,

    # Reproducibility
    "seed": 42

}

np.random.seed(
    MODEL_CONFIG["seed"]
)

torch.manual_seed(
    MODEL_CONFIG["seed"]
)

print_header("KONFIGURASI SHAP")

print()

for key, value in MODEL_CONFIG.items():

    print_info(f"{key} : {value}")

print()

print_success(
    "Konfigurasi SHAP berhasil dibuat."
)

KONFIGURASI SHAP

ℹ️ max_length : 128
ℹ️ batch_size : 8
ℹ️ seed : 42

✅ Konfigurasi SHAP berhasil dibuat.


## 4.3 Fungsi Inferensi Model

In [ ]:
# =====================================================
# CELL 16 : FUNGSI INFERENSI MODEL
# =====================================================

def predict_logits(texts):
    """
    Menghasilkan logits dari model IndoBERT.
    Fungsi ini menjadi dasar seluruh proses inferensi.
    """

    encoded = tokenizer(

        list(texts),

        padding=True,

        truncation=True,

        max_length=MODEL_CONFIG["max_length"],

        return_tensors="pt"

    )

    encoded = {

        key: value.to(DEVICE)

        for key, value in encoded.items()

    }

    with torch.no_grad():

        logits = model(
            **encoded
        ).logits

    return logits.cpu().numpy()


def predict_proba(texts):
    """
    Menghasilkan probabilitas prediksi.
    Digunakan untuk evaluasi model dan LIME.
    """

    logits = predict_logits(texts)

    logits = torch.tensor(logits)

    probability = torch.softmax(
        logits,
        dim=1
    )

    return probability.numpy()


def predict_logit(texts):
    """
    Menghasilkan log-odds (logit) untuk SHAP.
    """

    probability = predict_proba(texts)

    epsilon = 1e-8

    probability = np.clip(
        probability,
        epsilon,
        1 - epsilon
    )

    return np.log(
        probability / (1 - probability)
    )


print_header("FUNGSI INFERENSI MODEL")

print()

print_success(
    "Fungsi inferensi berhasil dibuat."
)

FUNGSI INFERENSI MODEL

✅ Fungsi inferensi berhasil dibuat.


## 4.4 Validasi Fungsi Inferensi

In [ ]:
# =====================================================
# CELL 17 : VALIDASI FUNGSI INFERENSI
# =====================================================

print_header("VALIDASI FUNGSI INFERENSI")

sample_text = [

    selected_reviews_df.loc[
        0,
        "text"
    ]

]

probability = predict_proba(
    sample_text
)

logit = predict_logit(
    sample_text
)

print()

print("Probabilitas")

display(

    pd.DataFrame(

        probability,

        columns=[

            LABEL_NAMES[0],

            LABEL_NAMES[1],

            LABEL_NAMES[2]

        ]

    )

)

print()

print("Logit")

display(

    pd.DataFrame(

        logit,

        columns=[

            LABEL_NAMES[0],

            LABEL_NAMES[1],

            LABEL_NAMES[2]

        ]

    )

)

print()

print_success(
    "Fungsi inferensi berhasil divalidasi."
)

VALIDASI FUNGSI INFERENSI

Probabilitas


,Negatif,Netral,Positif
0,0.000731,0.001093,0.998176



Logit


,Negatif,Netral,Positif
0,-7.220282,-6.817388,6.304689



✅ Fungsi inferensi berhasil divalidasi.


## 4.5 Membuat Text Masker

In [ ]:
# =====================================================
# CELL 18 : MEMBUAT TEXT MASKER
# =====================================================

print_header("MEMBUAT TEXT MASKER")

masker = shap.maskers.Text(
    tokenizer
)

print()

print_success(
    "Text masker berhasil dibuat."
)

MEMBUAT TEXT MASKER

✅ Text masker berhasil dibuat.


## 4.6 Validasi Tokenizer

In [ ]:
# =====================================================
# CELL 19 : VALIDASI TOKENIZER
# =====================================================

print_header("VALIDASI TOKENIZER")

sample_text = selected_reviews_df.loc[
    0,
    "text"
]

tokens = tokenizer.tokenize(
    sample_text
)

token_ids = tokenizer.convert_tokens_to_ids(
    tokens
)

tokenizer_df = pd.DataFrame({

    "Token": tokens,

    "Token ID": token_ids

})

print()

print_info(f"Sample ID : {selected_reviews_df.loc[0,'sample_id']}")

print_info(f"Text : {sample_text}")

print()

display(
    tokenizer_df
)

print()

print_info(
    f"Jumlah Token : {len(tokens)}"
)

print()

print_success(
    "Tokenizer berhasil divalidasi."
)

VALIDASI TOKENIZER

ℹ️ Sample ID : XAI_001
ℹ️ Text : sangat membantu dan mudah



,Token,Token ID
0,sangat,310
1,membantu,1055
2,dan,41
3,mudah,783



ℹ️ Jumlah Token : 4

✅ Tokenizer berhasil divalidasi.


## 4.7 Membangun SHAP Explainer

In [ ]:
# =====================================================
# CELL 20 : MEMBUAT SHAP EXPLAINER
# =====================================================

print_header("MEMBUAT SHAP EXPLAINER")

explainer = shap.Explainer(

    predict_logit,

    masker,

    output_names=[

        LABEL_NAMES[0],

        LABEL_NAMES[1],

        LABEL_NAMES[2]

    ]

)

print()

print_info(
    f"Explainer : {type(explainer).__name__}"
)

print_info(
    f"Jumlah Label : {len(LABEL_NAMES)}"
)

print()

print_success(
    "SHAP Explainer berhasil dibuat."
)

MEMBUAT SHAP EXPLAINER

ℹ️ Explainer : PartitionExplainer
ℹ️ Jumlah Label : 3

✅ SHAP Explainer berhasil dibuat.


## 4.8 Validasi SHAP Explainer

In [ ]:
# =====================================================
# CELL 21 : VALIDASI SHAP EXPLAINER
# =====================================================

print_header("VALIDASI SHAP EXPLAINER")

sample_text = [

    selected_reviews_df.loc[
        0,
        "text"
    ]
]

print()

print_info(
    f"Sample ID : {selected_reviews_df.loc[0,'sample_id']}"
)

print_info(
    f"Text : {sample_text[0]}"
)

print()

sample_shap = explainer(
    sample_text
)

print_success(
    "SHAP values berhasil dihitung."
)

print()

print_info(
    f"Tipe Output : {type(sample_shap).__name__}"
)

print_info(
    f"Jumlah Sampel : {len(sample_shap)}"
)

print_info(
    f"Jumlah Label : {len(LABEL_NAMES)}"
)

print()

print_success(
    "Explainer berhasil divalidasi."
)

VALIDASI SHAP EXPLAINER

ℹ️ Sample ID : XAI_001
ℹ️ Text : sangat membantu dan mudah

✅ SHAP values berhasil dihitung.

ℹ️ Tipe Output : Explanation
ℹ️ Jumlah Sampel : 1
ℹ️ Jumlah Label : 3

✅ Explainer berhasil divalidasi.


# 5. Persiapan Analisis SHAP

## 5.1 Membuat Struktur Folder SHAP

In [ ]:
# =====================================================
# CELL : KONFIGURASI GLOBAL
# =====================================================

print_header("KONFIGURASI GLOBAL")

# Root Project
PROJECT_DIR = "/content/drive/MyDrive/Skripsi_IndoBERT"

# Model
MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "model_indobert_terbaik"
)

# Dataset
SELECTED_DATA_PATH = os.path.join(
    PROJECT_DIR,
    "xai",
    "XB_Persiapan_XAI",
    "csv",
    "selected_reviews_xai.csv"
)

SAMPLING_LOG_PATH = os.path.join(
    PROJECT_DIR,
    "xai",
    "XB_Persiapan_XAI",
    "log",
    "sampling_log.csv"
)

print()
print_info(f"PROJECT_DIR : {PROJECT_DIR}")
print_info(f"MODEL_DIR   : {MODEL_DIR}")
print_info(f"DATASET     : {SELECTED_DATA_PATH}")
print()
print_success("Konfigurasi global berhasil dibuat.")

KONFIGURASI GLOBAL

ℹ️ PROJECT_DIR : /content/drive/MyDrive/Skripsi_IndoBERT
ℹ️ MODEL_DIR   : /content/drive/MyDrive/Skripsi_IndoBERT/model_indobert_terbaik
ℹ️ DATASET     : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/csv/selected_reviews_xai.csv

✅ Konfigurasi global berhasil dibuat.


In [ ]:
# =====================================================
# CELL 22 : MEMBUAT STRUKTUR FOLDER SHAP
# =====================================================

print_header("MEMBUAT STRUKTUR FOLDER SHAP")

# Root Output
XC_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "xai",
    "XC_SHAP_Analysis"
)

# Folder utama
CSV_DIR = os.path.join(
    XC_OUTPUT_DIR,
    "csv"
)

LOCAL_DIR = os.path.join(
    XC_OUTPUT_DIR,
    "local"
)

GLOBAL_DIR = os.path.join(
    XC_OUTPUT_DIR,
    "global"
)

OVERALL_DIR = os.path.join(
    XC_OUTPUT_DIR,
    "overall"
)

LOG_DIR = os.path.join(
    XC_OUTPUT_DIR,
    "log"
)

README_DIR = os.path.join(
    XC_OUTPUT_DIR,
    "readme"
)

# Subfolder Local
LOCAL_TEXT_DIR = os.path.join(
    LOCAL_DIR,
    "text_plot"
)

LOCAL_BAR_DIR = os.path.join(
    LOCAL_DIR,
    "bar_plot"
)

LOCAL_WATERFALL_DIR = os.path.join(
    LOCAL_DIR,
    "waterfall_plot"
)

LOCAL_METADATA_DIR = os.path.join(
    LOCAL_DIR,
    "metadata"
)

# Subfolder Global
GLOBAL_HIGH_DIR = os.path.join(
    GLOBAL_DIR,
    "high_confidence"
)

GLOBAL_LOW_DIR = os.path.join(
    GLOBAL_DIR,
    "low_confidence"
)

GLOBAL_ERROR_DIR = os.path.join(
    GLOBAL_DIR,
    "error_analysis"
)

folders = [

    XC_OUTPUT_DIR,

    CSV_DIR,

    LOCAL_DIR,

    GLOBAL_DIR,

    OVERALL_DIR,

    LOG_DIR,

    README_DIR,

    LOCAL_TEXT_DIR,

    LOCAL_BAR_DIR,

    LOCAL_WATERFALL_DIR,

    LOCAL_METADATA_DIR,

    GLOBAL_HIGH_DIR,

    GLOBAL_LOW_DIR,

    GLOBAL_ERROR_DIR

]

for folder in folders:

    os.makedirs(
        folder,
        exist_ok=True
    )

print()

for folder in folders:

    print_success(folder)

print()

print_success(
    "Seluruh struktur folder SHAP berhasil dibuat."
)

MEMBUAT STRUKTUR FOLDER SHAP

✅ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis
✅ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/csv
✅ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/local
✅ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/global
✅ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/overall
✅ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/log
✅ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/readme
✅ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/local/text_plot
✅ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/local/bar_plot
✅ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/local/waterfall_plot
✅ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/local/metadata
✅ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/global/high_confidence
✅ /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/global/low_confidence
✅ /

## 5.2 Inisialisasi Penyimpanan Hasil SHAP

In [ ]:
# =====================================================
# CELL 23 : INISIALISASI PENYIMPANAN SHAP
# =====================================================

print_header("INISIALISASI PENYIMPANAN SHAP")

SHAP_RESULTS = {}

LOCAL_SUMMARY = []

GLOBAL_SUMMARY = {}

OVERALL_SUMMARY = {}

print()

print_info(
    f"Jumlah Sampel : {len(selected_reviews_df)}"
)

print_info(
    f"Objek SHAP_RESULTS : {type(SHAP_RESULTS).__name__}"
)

print()

print_success(
    "Struktur penyimpanan SHAP berhasil dibuat."
)

INISIALISASI PENYIMPANAN SHAP

ℹ️ Jumlah Sampel : 30
ℹ️ Objek SHAP_RESULTS : dict

✅ Struktur penyimpanan SHAP berhasil dibuat.


## 5.3 Menghasilkan SHAP Values untuk Satu Sampel

In [ ]:
# =====================================================
# CELL 24 : SHAP VALUES SATU SAMPEL
# =====================================================

print_header("MENGHASILKAN SHAP VALUES")

sample_row = selected_reviews_df.iloc[0]

sample_id = sample_row["sample_id"]

sample_text = sample_row["text"]

print()

print_info(f"Sample ID : {sample_id}")

print_info(f"Text : {sample_text}")

print()

shap_result = explainer([sample_text])

SHAP_RESULTS[sample_id] = {

    "sample_id": sample_id,

    "text": sample_text,

    "actual_label": sample_row["actual_label"],

    "actual_sentiment": sample_row["actual_sentiment"],

    "predicted_label": sample_row["predicted_label"],

    "predicted_sentiment": sample_row["predicted_sentiment"],

    "prediction_case": sample_row["prediction_case"],

    "confidence": sample_row["confidence"],

    "sample_group": sample_row["sample_group"],

    "selection_reason": sample_row["selection_reason"],

    "shap_values": shap_result

}

print_success(
    "SHAP values berhasil dihitung."
)

print()

print_success(
    "Hasil berhasil disimpan ke SHAP_RESULTS."
)

MENGHASILKAN SHAP VALUES

ℹ️ Sample ID : XAI_001
ℹ️ Text : sangat membantu dan mudah

✅ SHAP values berhasil dihitung.

✅ Hasil berhasil disimpan ke SHAP_RESULTS.


## 5.4 Validasi SHAP Values

In [ ]:
# =====================================================
# CELL 25 : VALIDASI SHAP VALUES
# =====================================================

print_header("VALIDASI SHAP VALUES")

sample_data = SHAP_RESULTS["XAI_001"]

print()

print_info(f"Sample ID : {sample_data['sample_id']}")

print_info(f"Actual : {sample_data['actual_sentiment']}")

print_info(f"Prediksi : {sample_data['predicted_sentiment']}")

print_info(f"Confidence : {sample_data['confidence']:.6f}")

print_info(f"Group : {sample_data['sample_group']}")

print()

print("Shape SHAP Values")

print(sample_data["shap_values"].values.shape)

print()

print_success(
    "SHAP values berhasil divalidasi."
)

VALIDASI SHAP VALUES

ℹ️ Sample ID : XAI_001
ℹ️ Actual : Positif
ℹ️ Prediksi : Positif
ℹ️ Confidence : 0.998172
ℹ️ Group : High Confidence

Shape SHAP Values
(1, 6, 3)

✅ SHAP values berhasil divalidasi.


## 5.5 Analisis Statistik Token

In [ ]:
# =====================================================
# CELL 26 : ANALISIS STATISTIK TOKEN
# =====================================================

print_header("ANALISIS STATISTIK TOKEN")

sample_data = SHAP_RESULTS["XAI_001"]

explanation = sample_data["shap_values"]

predicted_class = sample_data["predicted_label"]

# ==========================================
# Ambil Token dan SHAP Value
# ==========================================

tokens = explanation.data[0]

token_scores = explanation.values[0, :, predicted_class]

token_df = pd.DataFrame({

    "Token": tokens,

    "SHAP Value": token_scores

})

# ==========================================
# Bersihkan Token
# ==========================================

token_df["Token"] = token_df["Token"].astype(str)

SPECIAL_TOKENS = {

    "",

    "[CLS]",

    "[SEP]",

    "[PAD]",

    "[MASK]"

}

token_df = token_df[
    token_df["Token"].str.strip() != ""
]

token_df = token_df[
    ~token_df["Token"].isin(SPECIAL_TOKENS)
]

token_df = token_df.reset_index(drop=True)

# ==========================================
# Statistik Token
# ==========================================

token_df["|SHAP|"] = token_df["SHAP Value"].abs()

token_df["Direction"] = np.where(

    token_df["SHAP Value"] >= 0,

    "Positif",

    "Negatif"

)

token_df = token_df.sort_values(

    by="|SHAP|",

    ascending=False

).reset_index(drop=True)

# ==========================================
# Token Positif & Negatif
# ==========================================

positive_df = token_df[
    token_df["SHAP Value"] > 0
].copy()

negative_df = token_df[
    token_df["SHAP Value"] < 0
].copy()

top_positive_token = None
top_positive_value = None

if len(positive_df) > 0:

    idx = positive_df["SHAP Value"].idxmax()

    top_positive_token = positive_df.loc[idx, "Token"]

    top_positive_value = float(
        positive_df.loc[idx, "SHAP Value"]
    )

top_negative_token = None
top_negative_value = None

if len(negative_df) > 0:

    idx = negative_df["SHAP Value"].idxmin()

    top_negative_token = negative_df.loc[idx, "Token"]

    top_negative_value = float(
        negative_df.loc[idx, "SHAP Value"]
    )

# ==========================================
# Simpan Statistik
# ==========================================

sample_data["statistics"] = {

    "token_table": token_df,

    "top_token": token_df.iloc[0]["Token"],

    "top_shap": float(
        token_df.iloc[0]["SHAP Value"]
    ),

    "mean_abs_shap": float(
        token_df["|SHAP|"].mean()
    ),

    "max_abs_shap": float(
        token_df["|SHAP|"].max()
    ),

    "num_tokens": int(
        len(token_df)
    ),

    "num_positive_tokens": int(
        len(positive_df)
    ),

    "num_negative_tokens": int(
        len(negative_df)
    ),

    "top_positive_token": top_positive_token,

    "top_positive_value": top_positive_value,

    "top_negative_token": top_negative_token,

    "top_negative_value": top_negative_value

}

# ==========================================
# Tampilkan Hasil
# ==========================================

print()

display(token_df)

print()

print_info(
    f"Jumlah Token : {len(token_df)}"
)

print_info(
    f"Top Token : {sample_data['statistics']['top_token']}"
)

print_info(
    f"Top Positive Token : {top_positive_token}"
)

print_info(
    f"Top Negative Token : {top_negative_token}"
)

print_info(
    f"Mean |SHAP| : {sample_data['statistics']['mean_abs_shap']:.6f}"
)

print()

print_success(
    "Statistik token berhasil dibuat."
)

ANALISIS STATISTIK TOKEN



,Token,SHAP Value,|SHAP|,Direction
0,mudah,3.954137,3.954137,Positif
1,membantu,3.049329,3.049329,Positif
2,dan,1.629414,1.629414,Positif
3,sangat,0.833540,0.833540,Positif



ℹ️ Jumlah Token : 4
ℹ️ Top Token : mudah
ℹ️ Top Positive Token : mudah
ℹ️ Top Negative Token : None
ℹ️ Mean |SHAP| : 2.366605

✅ Statistik token berhasil dibuat.


## 5.6 Validasi Statistik Token

In [ ]:
# =====================================================
# CELL 27 : VALIDASI STATISTIK TOKEN
# =====================================================

print_header("VALIDASI STATISTIK TOKEN")

stats = SHAP_RESULTS["XAI_001"]["statistics"]

print()

print_info(
    f"Jumlah Token : {stats['num_tokens']}"
)

print_info(
    f"Token Positif : {stats['num_positive_tokens']}"
)

print_info(
    f"Token Negatif : {stats['num_negative_tokens']}"
)

print()

print_info(
    f"Top Token : {stats['top_token']}"
)

print_info(
    f"Top Positive Token : {stats['top_positive_token']}"
)

print_info(
    f"Top Negative Token : {stats['top_negative_token']}"
)

print()

print_info(
    f"Mean |SHAP| : {stats['mean_abs_shap']:.6f}"
)

print_info(
    f"Max |SHAP| : {stats['max_abs_shap']:.6f}"
)

print()

print_success(
    "Statistik token berhasil divalidasi."
)

VALIDASI STATISTIK TOKEN

ℹ️ Jumlah Token : 4
ℹ️ Token Positif : 4
ℹ️ Token Negatif : 0

ℹ️ Top Token : mudah
ℹ️ Top Positive Token : mudah
ℹ️ Top Negative Token : None

ℹ️ Mean |SHAP| : 2.366605
ℹ️ Max |SHAP| : 3.954137

✅ Statistik token berhasil divalidasi.


## 5.7 Fungsi Waterfall Plot

In [ ]:
# =====================================================
# CELL 28 : FUNGSI WATERFALL PLOT
# =====================================================

print_header("MEMBUAT FUNGSI WATERFALL PLOT")

def save_waterfall_plot(
    sample_id,
    sample_data,
    output_dir,
    max_display=10
):
    """
    Membuat dan menyimpan Waterfall Plot SHAP
    untuk satu sampel.
    """

    explanation = sample_data["shap_values"]

    predicted_class = sample_data["predicted_label"]

    output_path = os.path.join(

        output_dir,

        f"{sample_id}_waterfall.png"

    )

    plt.figure(figsize=(10, 6))

    shap.plots.waterfall(

        explanation[0, :, predicted_class],

        max_display=max_display,

        show=False

    )

    plt.savefig(

        output_path,

        dpi=300,

        bbox_inches="tight"

    )

    plt.close()

    sample_data.setdefault("plots", {})

    sample_data["plots"]["waterfall"] = output_path

    return output_path


print_success(
    "Fungsi Waterfall Plot berhasil dibuat."
)

MEMBUAT FUNGSI WATERFALL PLOT
✅ Fungsi Waterfall Plot berhasil dibuat.


## 5.8 Validasi Waterfall Plot

In [ ]:
# =====================================================
# CELL 29 : VALIDASI WATERFALL PLOT
# =====================================================

print_header("VALIDASI WATERFALL PLOT")

sample_id = "XAI_001"

sample_data = SHAP_RESULTS[sample_id]

output_path = save_waterfall_plot(

    sample_id=sample_id,

    sample_data=sample_data,

    output_dir=LOCAL_WATERFALL_DIR,

    max_display=10

)

print()

print_info(
    f"Sample ID : {sample_id}"
)

print_info(
    f"Output File : {os.path.basename(output_path)}"
)

print_info(
    f"Lokasi : {output_path}"
)

print()

if os.path.exists(output_path):

    print_success(
        "Waterfall Plot berhasil disimpan."
    )

else:

    raise FileNotFoundError(output_path)

VALIDASI WATERFALL PLOT

ℹ️ Sample ID : XAI_001
ℹ️ Output File : XAI_001_waterfall.png
ℹ️ Lokasi : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/local/waterfall_plot/XAI_001_waterfall.png

✅ Waterfall Plot berhasil disimpan.


## 5.9 Fungsi Bar Plot

In [ ]:
# =====================================================
# CELL 30 : FUNGSI BAR PLOT
# =====================================================

print_header("MEMBUAT FUNGSI BAR PLOT")

def save_bar_plot(
    sample_id,
    sample_data,
    output_dir,
    max_display=10
):
    """
    Membuat dan menyimpan SHAP Bar Plot
    untuk satu sampel.
    """

    explanation = sample_data["shap_values"]

    predicted_class = sample_data["predicted_label"]

    output_path = os.path.join(

        output_dir,

        f"{sample_id}_bar.png"

    )

    plt.figure(figsize=(8,6))

    shap.plots.bar(

        explanation[0, :, predicted_class],

        max_display=max_display,

        show=False

    )

    plt.savefig(

        output_path,

        dpi=300,

        bbox_inches="tight"

    )

    plt.close()

    sample_data.setdefault("plots", {})

    sample_data["plots"]["bar"] = output_path

    return output_path


print_success(
    "Fungsi Bar Plot berhasil dibuat."
)

MEMBUAT FUNGSI BAR PLOT
✅ Fungsi Bar Plot berhasil dibuat.


## 5.10 Validasi Bar Plot

In [ ]:
# =====================================================
# CELL 31 : VALIDASI BAR PLOT
# =====================================================

print_header("VALIDASI BAR PLOT")

sample_id = "XAI_001"

sample_data = SHAP_RESULTS[sample_id]

output_path = save_bar_plot(

    sample_id=sample_id,

    sample_data=sample_data,

    output_dir=LOCAL_BAR_DIR,

    max_display=10

)

print()

print_info(
    f"Sample ID : {sample_id}"
)

print_info(
    f"Output File : {os.path.basename(output_path)}"
)

print_info(
    f"Lokasi : {output_path}"
)

print()

if os.path.exists(output_path):

    print_success(
        "Bar Plot berhasil disimpan."
    )

else:

    raise FileNotFoundError(output_path)

VALIDASI BAR PLOT

ℹ️ Sample ID : XAI_001
ℹ️ Output File : XAI_001_bar.png
ℹ️ Lokasi : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/local/bar_plot/XAI_001_bar.png

✅ Bar Plot berhasil disimpan.


## 5.11 Fungsi Text Plot

In [ ]:
# =====================================================
# CELL 32 : FUNGSI TEXT PLOT
# =====================================================

print_header("MEMBUAT FUNGSI TEXT PLOT")

from IPython.display import HTML

def show_text_plot(
    sample_id,
    sample_data
):
    """
    Menampilkan SHAP Text Plot
    untuk satu sampel.
    """

    explanation = sample_data["shap_values"]

    predicted_class = sample_data["predicted_label"]

    text_plot = shap.plots.text(

        explanation[0, :, predicted_class],

        display=False

    )

    sample_data.setdefault("plots", {})

    sample_data["plots"]["text"] = "Displayed in Notebook"

    return HTML(text_plot)


print_success(
    "Fungsi Text Plot berhasil dibuat."
)

MEMBUAT FUNGSI TEXT PLOT
✅ Fungsi Text Plot berhasil dibuat.


## 5.12 Validasi Text Plot

In [ ]:
# =====================================================
# CELL 33 : VALIDASI TEXT PLOT
# =====================================================

print_header("VALIDASI TEXT PLOT")

sample_id = "XAI_001"

sample_data = SHAP_RESULTS[sample_id]

print()

print_info(
    f"Sample ID : {sample_id}"
)

print_info(
    f"Prediksi : {sample_data['predicted_sentiment']}"
)

print()

display(

    show_text_plot(

        sample_id,

        sample_data

    )

)

print()

print_success(
    "Text Plot berhasil ditampilkan."
)

VALIDASI TEXT PLOT

ℹ️ Sample ID : XAI_001
ℹ️ Prediksi : Positif




✅ Text Plot berhasil ditampilkan.


# 6. SHAP Local Analysis

## 6.1 Membuat Folder Output SHAP Local

In [ ]:
# =====================================================
# CELL 34 : MEMBUAT FOLDER OUTPUT SHAP LOCAL
# =====================================================

print_header("MEMBUAT FOLDER OUTPUT SHAP LOCAL")

LOCAL_METADATA_DIR = os.path.join(

    LOCAL_DIR,

    "metadata"

)

os.makedirs(

    LOCAL_METADATA_DIR,

    exist_ok=True

)

print()

print_info(

    f"Waterfall : {LOCAL_WATERFALL_DIR}"

)

print_info(

    f"Bar Plot  : {LOCAL_BAR_DIR}"

)

print_info(

    f"Text Plot : {LOCAL_TEXT_DIR}"

)

print_info(

    f"Metadata  : {LOCAL_METADATA_DIR}"

)

print()

print_success(

    "Folder SHAP Local berhasil divalidasi."

)

MEMBUAT FOLDER OUTPUT SHAP LOCAL

ℹ️ Waterfall : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/local/waterfall_plot
ℹ️ Bar Plot  : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/local/bar_plot
ℹ️ Text Plot : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/local/text_plot
ℹ️ Metadata  : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/local/metadata

✅ Folder SHAP Local berhasil divalidasi.


## 6.2 Menjalankan SHAP Local Seluruh Sampel

In [ ]:
# =====================================================
# CELL 35 : GENERATE SHAP LOCAL
# =====================================================

print_header("GENERATE SHAP LOCAL")

success_count = 0

failed_count = 0

failed_samples = []

for sample_id in tqdm(

    SHAP_RESULTS.keys(),

    desc="Generate SHAP Local"

):

    try:

        sample_data = SHAP_RESULTS[sample_id]

        # ------------------------------------
        # Waterfall
        # ------------------------------------

        save_waterfall_plot(

            sample_id,

            sample_data,

            LOCAL_WATERFALL_DIR

        )

        # ------------------------------------
        # Bar Plot
        # ------------------------------------

        save_bar_plot(

            sample_id,

            sample_data,

            LOCAL_BAR_DIR

        )

        # ------------------------------------
        # Text Plot
        # ------------------------------------

        show_text_plot(

            sample_id,

            sample_data

        )

        success_count += 1

    except Exception as e:

        failed_count += 1

        failed_samples.append(

            {

                "sample_id": sample_id,

                "error": str(e)

            }

        )

print()

print_info(

    f"Berhasil : {success_count}"

)

print_info(

    f"Gagal : {failed_count}"

)

print()

print_success(

    "Generate SHAP Local selesai."

)

GENERATE SHAP LOCAL


Generate SHAP Local:   0%|          | 0/1 [00:00<?, ?it/s]


ℹ️ Berhasil : 1
ℹ️ Gagal : 0

✅ Generate SHAP Local selesai.


## 6.3 Validasi Hasil SHAP Local

In [ ]:
# =====================================================
# CELL 36 : VALIDASI SHAP LOCAL
# =====================================================

print_header("VALIDASI SHAP LOCAL")

waterfall_total = len(

    os.listdir(LOCAL_WATERFALL_DIR)

)

bar_total = len(

    os.listdir(LOCAL_BAR_DIR)

)

print()

print_info(

    f"Waterfall Plot : {waterfall_total}"

)

print_info(

    f"Bar Plot : {bar_total}"

)

print_info(

    f"Jumlah Sampel : {len(SHAP_RESULTS)}"

)

print()

if (

    waterfall_total == len(SHAP_RESULTS)

    and

    bar_total == len(SHAP_RESULTS)

):

    print_success(

        "Seluruh SHAP Local berhasil dibuat."

    )

else:

    print_warning(

        "Masih terdapat output yang belum lengkap."

    )

VALIDASI SHAP LOCAL

ℹ️ Waterfall Plot : 30
ℹ️ Bar Plot : 30
ℹ️ Jumlah Sampel : 1

⚠️ Masih terdapat output yang belum lengkap.


## 6.4 Fungsi Generate SHAP Local

In [ ]:
# =====================================================
# CELL 37 : FUNGSI GENERATE SHAP LOCAL
# =====================================================

print_header("MEMBUAT FUNGSI GENERATE SHAP LOCAL")


def generate_local_explanation(
    sample_id,
    sample_data
):
    """
    Generate seluruh output SHAP Local
    untuk satu sampel.
    """

    # -------------------------
    # Waterfall Plot
    # -------------------------

    waterfall_path = save_waterfall_plot(

        sample_id=sample_id,

        sample_data=sample_data,

        output_dir=LOCAL_WATERFALL_DIR

    )

    # -------------------------
    # Bar Plot
    # -------------------------

    bar_path = save_bar_plot(

        sample_id=sample_id,

        sample_data=sample_data,

        output_dir=LOCAL_BAR_DIR

    )

    # -------------------------
    # Text Plot
    # -------------------------

    text_html = show_text_plot(

        sample_id,

        sample_data

    )

    # -------------------------
    # Simpan metadata plot
    # -------------------------

    sample_data.setdefault("plots", {})

    sample_data["plots"].update({

        "waterfall": waterfall_path,

        "bar": bar_path,

        "text": "Notebook Display"

    })

    return sample_data


print_success(
    "Fungsi Generate SHAP Local berhasil dibuat."
)

MEMBUAT FUNGSI GENERATE SHAP LOCAL
✅ Fungsi Generate SHAP Local berhasil dibuat.


## 6.5 Validasi Fungsi Generate SHAP Local

In [ ]:
# =====================================================
# CELL 38 : VALIDASI GENERATE SHAP LOCAL
# =====================================================

print_header("VALIDASI GENERATE SHAP LOCAL")

sample_id = "XAI_001"

sample_data = SHAP_RESULTS[sample_id]

sample_data = generate_local_explanation(

    sample_id,

    sample_data

)

print()

print_info(
    f"Sample ID : {sample_id}"
)

print_info(
    f"Waterfall : {os.path.basename(sample_data['plots']['waterfall'])}"
)

print_info(
    f"Bar Plot : {os.path.basename(sample_data['plots']['bar'])}"
)

print_info(
    f"Text Plot : {sample_data['plots']['text']}"
)

print()

print_success(
    "Generate SHAP Local berhasil divalidasi."
)

VALIDASI GENERATE SHAP LOCAL

ℹ️ Sample ID : XAI_001
ℹ️ Waterfall : XAI_001_waterfall.png
ℹ️ Bar Plot : XAI_001_bar.png
ℹ️ Text Plot : Notebook Display

✅ Generate SHAP Local berhasil divalidasi.


## 6.6 Generate SHAP Local Seluruh Sampel

In [ ]:
# =====================================================
# CELL 39 : GENERATE SHAP LOCAL 30 SAMPEL
# =====================================================

print_header("GENERATE SHAP LOCAL SELURUH SAMPEL")

success_count = 0

failed_count = 0

failed_samples = []

for sample_id, sample_data in tqdm(

    SHAP_RESULTS.items(),

    desc="Generate SHAP Local"

):

    try:

        # --------------------------
        # Waterfall
        # --------------------------

        save_waterfall_plot(

            sample_id,

            sample_data,

            LOCAL_WATERFALL_DIR

        )

        # --------------------------
        # Bar
        # --------------------------

        save_bar_plot(

            sample_id,

            sample_data,

            LOCAL_BAR_DIR

        )

        success_count += 1

    except Exception as e:

        failed_count += 1

        failed_samples.append({

            "sample_id": sample_id,

            "error": str(e)

        })

print()

print_info(f"Berhasil : {success_count}")

print_info(f"Gagal : {failed_count}")

if failed_count > 0:

    print()

    failed_df = pd.DataFrame(

        failed_samples

    )

    display(failed_df)

print()

print_success(

    "Generate SHAP Local selesai."

)

GENERATE SHAP LOCAL SELURUH SAMPEL


Generate SHAP Local:   0%|          | 0/1 [00:00<?, ?it/s]


ℹ️ Berhasil : 1
ℹ️ Gagal : 0

✅ Generate SHAP Local selesai.


## 6.7 Validasi Hasil SHAP Local

In [ ]:
# =====================================================
# CELL 40 : VALIDASI OUTPUT SHAP LOCAL
# =====================================================

print_header("VALIDASI OUTPUT SHAP LOCAL")

# =====================================================
# UTILITAS : HITUNG FILE
# =====================================================

def count_files(
    folder_path,
    extension=None
):
    """
    Menghitung jumlah file di dalam folder.

    Parameters
    ----------
    folder_path : str
        Lokasi folder.

    extension : str, optional
        Ekstensi file yang ingin dihitung,
        misalnya ".png" atau ".json".

    Returns
    -------
    int
        Jumlah file sesuai kriteria.
    """

    if extension is None:

        return len(os.listdir(folder_path))

    return len([

        file

        for file in os.listdir(folder_path)

        if file.endswith(extension)

    ])


# =====================================================
# VALIDASI OUTPUT
# =====================================================

waterfall_total = count_files(

    LOCAL_WATERFALL_DIR,

    ".png"

)

bar_total = count_files(

    LOCAL_BAR_DIR,

    ".png"

)

expected_total = len(

    SHAP_RESULTS

)

print()

print_info(

    f"Waterfall Plot : {waterfall_total}"

)

print_info(

    f"Bar Plot : {bar_total}"

)

print_info(

    f"Expected : {expected_total}"

)

print()

if (

    waterfall_total == expected_total

    and

    bar_total == expected_total

):

    print_success(

        "Seluruh output SHAP Local berhasil dibuat."

    )

else:

    print_warning(

        "Masih terdapat output yang belum lengkap."

    )

VALIDASI OUTPUT SHAP LOCAL

ℹ️ Waterfall Plot : 30
ℹ️ Bar Plot : 30
ℹ️ Expected : 1

⚠️ Masih terdapat output yang belum lengkap.


## 6.8 Generate Metadata SHAP Local

In [ ]:
# =====================================================
# CELL 41 : GENERATE METADATA SHAP LOCAL
# =====================================================

print_header("GENERATE METADATA SHAP LOCAL")

metadata_success = 0

for sample_id, sample_data in tqdm(

    SHAP_RESULTS.items(),

    desc="Generate Metadata"

):

    statistics = sample_data["statistics"]

    metadata = {

        "sample_id":
            sample_id,

        "actual_sentiment":
            sample_data["actual_sentiment"],

        "predicted_sentiment":
            sample_data["predicted_sentiment"],

        "confidence":
            float(sample_data["confidence"]),

        "sample_group":
            sample_data["sample_group"],

        "top_token":
            statistics["top_token"],

        "top_positive_token":
            statistics["top_positive_token"],

        "top_negative_token":
            statistics["top_negative_token"],

        "mean_abs_shap":
            float(statistics["mean_abs_shap"]),

        "max_abs_shap":
            float(statistics["max_abs_shap"]),

        "plots":
            sample_data["plots"]

    }

    output_path = os.path.join(

        LOCAL_METADATA_DIR,

        f"{sample_id}_metadata.json"

    )

    with open(

        output_path,

        "w",

        encoding="utf-8"

    ) as f:

        json.dump(

            metadata,

            f,

            ensure_ascii=False,

            indent=4

        )

    metadata_success += 1

print()

print_info(
    f"Metadata dibuat : {metadata_success}"
)

print()

print_success(
    "Seluruh metadata berhasil disimpan."
)

GENERATE METADATA SHAP LOCAL


Generate Metadata:   0%|          | 0/1 [00:00<?, ?it/s]


ℹ️ Metadata dibuat : 1

✅ Seluruh metadata berhasil disimpan.


## 6.9 Validasi Metadata SHAP Local

In [ ]:
# =====================================================
# CELL 42 : VALIDASI METADATA SHAP LOCAL
# =====================================================

print_header("VALIDASI METADATA SHAP LOCAL")

metadata_total = count_files(

    LOCAL_METADATA_DIR,

    ".json"

)

expected_total = len(

    SHAP_RESULTS

)

print()

print_info(

    f"Metadata JSON : {metadata_total}"

)

print_info(

    f"Expected : {expected_total}"

)

print()

if metadata_total == expected_total:

    print_success(

        "Seluruh metadata berhasil dibuat."

    )

else:

    print_warning(

        "Masih terdapat metadata yang belum lengkap."

    )

VALIDASI METADATA SHAP LOCAL

ℹ️ Metadata JSON : 30
ℹ️ Expected : 1

⚠️ Masih terdapat metadata yang belum lengkap.


## 6.10 Generate Summary SHAP Local

In [ ]:
# =====================================================
# CELL 43 : GENERATE SHAP EXPLANATION (30 SAMPEL)
# =====================================================

print_header("GENERATE SHAP EXPLANATION")

# Reset agar bersih apabila cell dijalankan ulang
SHAP_RESULTS = {}

success_count = 0

failed_count = 0

failed_samples = []

for _, row in tqdm(

    selected_reviews_df.iterrows(),

    total=len(selected_reviews_df),

    desc="Generate SHAP"

):

    sample_id = row["sample_id"]

    text = row["text"]

    try:

        explanation = explainer([text])

        predicted_label = int(

            np.argmax(

                predict_proba([text])[0]

            )

        )

        SHAP_RESULTS[sample_id] = {

            "sample_id": sample_id,

            "text": text,

            "actual_label": int(

                row["actual_label"]

            ),

            "actual_sentiment":

                row["actual_sentiment"],

            "predicted_label":

                predicted_label,

            "predicted_sentiment":

                row["predicted_sentiment"],

            "confidence":

                float(row["confidence"]),

            "sample_group":

                row["sample_group"],

            "prediction_case":

                row["prediction_case"],

            "confidence_level":

                row["confidence_level"],

            "selection_reason":

                row["selection_reason"],

            "shap_values":

                explanation

        }

        success_count += 1

    except Exception as e:

        failed_count += 1

        failed_samples.append({

            "sample_id": sample_id,

            "error": str(e)

        })

print()

print_info(

    f"Berhasil : {success_count}"

)

print_info(

    f"Gagal : {failed_count}"

)

if failed_count > 0:

    print()

    display(

        pd.DataFrame(

            failed_samples

        )

    )

print()

print_success(

    "SHAP Explanation seluruh sampel berhasil dibuat."

)

GENERATE SHAP EXPLANATION


Generate SHAP:   0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]


ℹ️ Berhasil : 30
ℹ️ Gagal : 0

✅ SHAP Explanation seluruh sampel berhasil dibuat.


## 6.11 Validasi Summary SHAP Local

In [ ]:
# =====================================================
# CELL 44 : VALIDASI SHAP RESULTS
# =====================================================

print_header("VALIDASI SHAP RESULTS")

print()

print_info(

    f"Jumlah SHAP_RESULTS : {len(SHAP_RESULTS)}"

)

print_info(

    f"Expected : {len(selected_reviews_df)}"

)

print()

missing_samples = [

    sid

    for sid in selected_reviews_df["sample_id"]

    if sid not in SHAP_RESULTS

]

if len(missing_samples) == 0:

    print_success(

        "Seluruh sampel berhasil memiliki SHAP Explanation."

    )

else:

    print_warning(

        f"Terdapat {len(missing_samples)} sampel yang belum diproses."

    )

    display(

        pd.DataFrame({

            "sample_id": missing_samples

        })

    )

VALIDASI SHAP RESULTS

ℹ️ Jumlah SHAP_RESULTS : 30
ℹ️ Expected : 30

✅ Seluruh sampel berhasil memiliki SHAP Explanation.


## 6.12 Generate Token Statistics Seluruh Sampel

In [ ]:
# =====================================================
# CELL 45A : EXPORT RAW SHAP MANUAL
# =====================================================

import os
import pandas as pd
import numpy as np

print("="*70)
print("EXPORT RAW SHAP UNTUK PERHITUNGAN MANUAL")
print("="*70)

SAVE_DIR = "/content/drive/MyDrive/Skripsi_IndoBERT/xai/XA_SHAP_Manual/RAW_SHAP"

os.makedirs(SAVE_DIR, exist_ok=True)

summary_rows = []

for sample_id, sample_data in SHAP_RESULTS.items():

    explanation = sample_data["shap_values"]

    pred_class = sample_data["predicted_label"]

    tokens = explanation.data[0]

    shap_values = explanation.values[0, :, pred_class]

    base_value = float(explanation.base_values[0, pred_class])

    df = pd.DataFrame({

        "token": tokens,

        "shap_value": shap_values

    })

    df = df[df["token"].astype(str).str.strip() != ""]

    df = df.reset_index(drop=True)

    raw_path = os.path.join(
        SAVE_DIR,
        f"{sample_id}.csv"
    )

    df.to_csv(raw_path, index=False)

    summary_rows.append({

        "sample_id": sample_id,

        "actual_label": sample_data["actual_sentiment"],

        "predicted_label": sample_data["predicted_sentiment"],

        "predicted_class": pred_class,

        "confidence": sample_data["confidence"],

        "base_value": base_value,

        "num_token": len(df)

    })

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(

    os.path.join(

        SAVE_DIR,

        "summary_raw_shap.csv"

    ),

    index=False

)

print()

print("Jumlah Sample :", len(summary_df))

print()

display(summary_df.head())

print()

print("Semua file berhasil disimpan.")

print()

print(SAVE_DIR)

EXPORT RAW SHAP UNTUK PERHITUNGAN MANUAL

Jumlah Sample : 30



,sample_id,actual_label,predicted_label,predicted_class,confidence,base_value,num_token
0,XAI_001,Positif,Positif,2,0.998172,-3.161730,4
1,XAI_002,Positif,Positif,2,0.998167,-3.772544,5
2,XAI_003,Positif,Positif,2,0.998162,-1.833358,3
3,XAI_004,Positif,Positif,2,0.998159,-3.161730,4
4,XAI_005,Positif,Positif,2,0.998155,-4.264643,6



Semua file berhasil disimpan.

/content/drive/MyDrive/Skripsi_IndoBERT/xai/XA_SHAP_Manual/RAW_SHAP


In [ ]:
# =====================================================
# CELL 45B : EXPORT LOGIT DAN VERIFIKASI SHAP
# =====================================================

import os
import torch
import numpy as np
import pandas as pd

print("="*70)
print("EXPORT LOGIT DAN VERIFIKASI SHAP")
print("="*70)

SAVE_DIR = "/content/drive/MyDrive/Skripsi_IndoBERT/xai/XA_SHAP_Manual/RAW_SHAP"
os.makedirs(SAVE_DIR, exist_ok=True)

rows = []

for sample_id, sample_data in SHAP_RESULTS.items():

    # =====================================================
    # Text
    # =====================================================

    text = sample_data["text"]

    # =====================================================
    # Tokenisasi
    # =====================================================

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    # =====================================================
    # Samakan device CPU/GPU
    # =====================================================

    device = next(model.parameters()).device

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    # =====================================================
    # Forward Model
    # =====================================================

    with torch.no_grad():

        outputs = model(**inputs)

    # =====================================================
    # Logit
    # =====================================================

    logits = outputs.logits.squeeze(0).detach().cpu().numpy()

    # =====================================================
    # Probability
    # =====================================================

    probs = (
        torch.softmax(outputs.logits, dim=1)
        .squeeze(0)
        .detach()
        .cpu()
        .numpy()
    )

    # =====================================================
    # SHAP
    # =====================================================

    explanation = sample_data["shap_values"]

    pred_class = sample_data["predicted_label"]

    base_value = float(
        explanation.base_values[0, pred_class]
    )

    shap_values = explanation.values[0, :, pred_class]

    shap_sum = float(
        np.sum(shap_values)
    )

    reconstructed = base_value + shap_sum

    # =====================================================
    # Simpan
    # =====================================================

    rows.append({

        "sample_id": sample_id,

        "actual_label": sample_data["actual_sentiment"],

        "prediction": sample_data["predicted_sentiment"],

        "predicted_class": pred_class,

        "confidence": float(sample_data["confidence"]),

        "base_value": base_value,

        "sum_shap": shap_sum,

        "base_plus_sum_shap": reconstructed,

        "logit_negatif": float(logits[0]),

        "logit_netral": float(logits[1]),

        "logit_positif": float(logits[2]),

        "prob_negatif": float(probs[0]),

        "prob_netral": float(probs[1]),

        "prob_positif": float(probs[2])

    })

# =====================================================
# DataFrame
# =====================================================

verify_df = pd.DataFrame(rows)

display(verify_df.head())

# =====================================================
# Simpan CSV
# =====================================================

csv_path = os.path.join(
    SAVE_DIR,
    "verification_shap.csv"
)

verify_df.to_csv(
    csv_path,
    index=False
)

print()

print("Jumlah Sampel :", len(verify_df))

print()

print("File berhasil disimpan :")

print(csv_path)

EXPORT LOGIT DAN VERIFIKASI SHAP


,sample_id,actual_label,prediction,predicted_class,confidence,base_value,sum_shap,base_plus_sum_shap,logit_negatif,logit_netral,logit_positif,prob_negatif,prob_netral,prob_positif
0,XAI_001,Positif,Positif,2,0.998172,-3.161730,9.466420,6.304689,-2.570307,-2.167776,4.648880,0.000731,0.001093,0.998176
1,XAI_002,Positif,Positif,2,0.998167,-3.772544,10.070871,6.298327,-2.579970,-2.148450,4.650320,0.000723,0.001113,0.998164
2,XAI_003,Positif,Positif,2,0.998162,-1.833358,8.135041,6.301683,-2.527651,-2.218322,4.633738,0.000775,0.001055,0.998170
3,XAI_004,Positif,Positif,2,0.998159,-3.161730,9.462500,6.300770,-2.500570,-2.232597,4.636285,0.000794,0.001038,0.998168
4,XAI_005,Positif,Positif,2,0.998155,-4.264643,10.564956,6.300313,-2.560487,-2.182172,4.639935,0.000745,0.001087,0.998168



Jumlah Sampel : 30

File berhasil disimpan :
/content/drive/MyDrive/Skripsi_IndoBERT/xai/XA_SHAP_Manual/RAW_SHAP/verification_shap.csv


In [ ]:
# =====================================================
# CELL 45C : VALIDASI OUTPUT PREDICT_LOGIT SHAP
# =====================================================

import numpy as np

print("="*70)
print("VALIDASI OUTPUT PREDICT_LOGIT")
print("="*70)

sample = SHAP_RESULTS["XAI_001"]

text = sample["text"]

# ---------------------------------------
# Output fungsi predict_logit
# ---------------------------------------

pred = predict_logit([text])

print("\nOutput predict_logit:")
print(pred)

# ---------------------------------------
# SHAP
# ---------------------------------------

exp = sample["shap_values"]

base = exp.base_values[0]

sum_shap = np.sum(exp.values[0], axis=0)

print("\nBase Value")
print(base)

print("\nSum SHAP")
print(sum_shap)

print("\nBase + Sum SHAP")
print(base + sum_shap)

print("\nSelisih")
print(pred[0] - (base + sum_shap))

VALIDASI OUTPUT PREDICT_LOGIT

Output predict_logit:
[[-7.2202816 -6.8173876  6.3046894]]

Base Value
[ 3.10183787 -6.02993536 -3.16173029]

Sum SHAP
[-10.32211947  -0.78745222   9.4664197 ]

Base + Sum SHAP
[-7.2202816  -6.81738758  6.30468941]

Selisih
[0. 0. 0.]


In [ ]:
sample = SHAP_RESULTS["XAI_001"]

exp = sample["shap_values"]

print("="*60)
print("BASE VALUES")
print("="*60)
print(exp.base_values)

print()

print("="*60)
print("JUMLAH SHAP PER KELAS")
print("="*60)

print(np.sum(exp.values[0,:,0]))
print(np.sum(exp.values[0,:,1]))
print(np.sum(exp.values[0,:,2]))

print()

print("="*60)
print("BASE + SHAP")
print("="*60)

print(exp.base_values[0,0] + np.sum(exp.values[0,:,0]))
print(exp.base_values[0,1] + np.sum(exp.values[0,:,1]))
print(exp.base_values[0,2] + np.sum(exp.values[0,:,2]))

BASE VALUES
[[ 3.10183787 -6.02993536 -3.16173029]]

JUMLAH SHAP PER KELAS
-10.32211947441101
-0.787452220916748
9.466419696807861

BASE + SHAP
-7.220281600952148
-6.817387580871582
6.304689407348633


In [ ]:
# =====================================================
# CELL 45 : GENERATE TOKEN STATISTICS (30 SAMPEL)
# =====================================================

print_header("GENERATE TOKEN STATISTICS")

statistics_success = 0

for sample_id, sample_data in tqdm(

    SHAP_RESULTS.items(),

    desc="Generate Statistics"

):

    explanation = sample_data["shap_values"]

    predicted_class = sample_data["predicted_label"]

    values = explanation.values[0, :, predicted_class]

    tokens = explanation.data[0]

    token_df = pd.DataFrame({

        "Token": tokens,

        "SHAP Value": values

    })

    token_df = token_df[

        token_df["Token"].astype(str).str.strip() != ""

    ].copy()

    token_df["|SHAP|"] = (

        token_df["SHAP Value"]

        .abs()

    )

    token_df["Direction"] = np.where(

        token_df["SHAP Value"] >= 0,

        "Positif",

        "Negatif"

    )

    token_df = token_df.sort_values(

        "|SHAP|",

        ascending=False

    ).reset_index(drop=True)

    positive_df = token_df[

        token_df["SHAP Value"] > 0

    ]

    negative_df = token_df[

        token_df["SHAP Value"] < 0

    ]

    # =====================================================
    # Simpan DataFrame token secara independen
    # =====================================================

    sample_data["token_statistics"] = token_df.copy()

    # =====================================================
    # Simpan ringkasan statistik
    # =====================================================

    sample_data["statistics"] = {

        "token_count":

            len(token_df),

        "top_token":

            token_df.iloc[0]["Token"],

        "top_positive_token":

            positive_df.iloc[0]["Token"]

            if len(positive_df) > 0

            else None,

        "top_negative_token":

            negative_df.iloc[0]["Token"]

            if len(negative_df) > 0

            else None,

        "mean_abs_shap":

            float(

                token_df["|SHAP|"].mean()

            ),

        "max_abs_shap":

            float(

                token_df["|SHAP|"].max()

            )

    }

    statistics_success += 1

print()

print_info(

    f"Statistik dibuat : {statistics_success}"

)

print()

print_success(

    "Token Statistics seluruh sampel berhasil dibuat."

)

GENERATE TOKEN STATISTICS


Generate Statistics:   0%|          | 0/30 [00:00<?, ?it/s]


ℹ️ Statistik dibuat : 30

✅ Token Statistics seluruh sampel berhasil dibuat.


## 6.13 Validasi Token Statistics

In [ ]:
# =====================================================
# CELL 46 : VALIDASI TOKEN STATISTICS
# =====================================================

print_header("VALIDASI TOKEN STATISTICS")

missing_statistics = []

for sample_id, sample_data in SHAP_RESULTS.items():

    if "statistics" not in sample_data:

        missing_statistics.append(sample_id)

print()

print_info(

    f"Expected : {len(SHAP_RESULTS)}"

)

print_info(

    f"Statistics : {len(SHAP_RESULTS)-len(missing_statistics)}"

)

print()

if len(missing_statistics) == 0:

    print_success(

        "Seluruh Token Statistics berhasil dibuat."

    )

else:

    print_warning(

        f"Terdapat {len(missing_statistics)} sampel yang belum memiliki statistik."

    )

    display(

        pd.DataFrame({

            "sample_id": missing_statistics

        })

    )

VALIDASI TOKEN STATISTICS

ℹ️ Expected : 30
ℹ️ Statistics : 30

✅ Seluruh Token Statistics berhasil dibuat.


## 6.14 Generate Waterfall Plot Seluruh Sampel

In [ ]:
# =====================================================
# CELL 47 : GENERATE WATERFALL PLOT (30 SAMPEL)
# =====================================================

print_header("GENERATE WATERFALL PLOT")

waterfall_success = 0
waterfall_failed = []

for sample_id, sample_data in tqdm(

    SHAP_RESULTS.items(),

    desc="Generate Waterfall"

):

    try:

        explanation = sample_data["shap_values"]

        predicted_class = sample_data["predicted_label"]

        output_path = os.path.join(

            LOCAL_WATERFALL_DIR,

            f"{sample_id}_waterfall.png"

        )

        plt.figure(figsize=(10,6))

        shap.plots.waterfall(

            explanation[0, :, predicted_class],

            max_display=10,

            show=False

        )

        plt.savefig(

            output_path,

            dpi=300,

            bbox_inches="tight"

        )

        plt.close()

        sample_data.setdefault(

            "plots",

            {}

        )

        sample_data["plots"]["waterfall"] = output_path

        waterfall_success += 1

    except Exception as e:

        waterfall_failed.append({

            "sample_id": sample_id,

            "error": str(e)

        })

print()

print_info(

    f"Berhasil : {waterfall_success}"

)

print_info(

    f"Gagal : {len(waterfall_failed)}"

)

if waterfall_failed:

    print()

    display(

        pd.DataFrame(

            waterfall_failed

        )

    )

print()

print_success(

    "Waterfall Plot seluruh sampel berhasil dibuat."

)

GENERATE WATERFALL PLOT


Generate Waterfall:   0%|          | 0/30 [00:00<?, ?it/s]


ℹ️ Berhasil : 30
ℹ️ Gagal : 0

✅ Waterfall Plot seluruh sampel berhasil dibuat.


## 6.15 Validasi Waterfall Plot

In [ ]:
# =====================================================
# CELL 48 : VALIDASI WATERFALL PLOT
# =====================================================

print_header("VALIDASI WATERFALL PLOT")

waterfall_total = count_files(

    LOCAL_WATERFALL_DIR,

    ".png"

)

print()

print_info(

    f"Waterfall Plot : {waterfall_total}"

)

print_info(

    f"Expected : {len(SHAP_RESULTS)}"

)

print()

if waterfall_total == len(SHAP_RESULTS):

    print_success(

        "Seluruh Waterfall Plot berhasil dibuat."

    )

else:

    print_warning(

        f"Masih terdapat {len(SHAP_RESULTS)-waterfall_total} file yang belum dibuat."

    )

VALIDASI WATERFALL PLOT

ℹ️ Waterfall Plot : 30
ℹ️ Expected : 30

✅ Seluruh Waterfall Plot berhasil dibuat.


## 6.16 Generate Bar Plot Seluruh Sampel

In [ ]:
# =====================================================
# CELL 49 : GENERATE BAR PLOT (30 SAMPEL)
# =====================================================

print_header("GENERATE BAR PLOT")

bar_success = 0

bar_failed = []

for sample_id, sample_data in tqdm(

    SHAP_RESULTS.items(),

    desc="Generate Bar Plot"

):

    try:

        explanation = sample_data["shap_values"]

        predicted_class = sample_data["predicted_label"]

        output_path = os.path.join(

            LOCAL_BAR_DIR,

            f"{sample_id}_bar.png"

        )

        plt.figure(figsize=(10,6))

        shap.plots.bar(

            explanation[0, :, predicted_class],

            max_display=10,

            show=False

        )

        plt.savefig(

            output_path,

            dpi=300,

            bbox_inches="tight"

        )

        plt.close()

        sample_data.setdefault(

            "plots",

            {}

        )

        sample_data["plots"]["bar"] = output_path

        bar_success += 1

    except Exception as e:

        bar_failed.append({

            "sample_id": sample_id,

            "error": str(e)

        })

print()

print_info(

    f"Berhasil : {bar_success}"

)

print_info(

    f"Gagal : {len(bar_failed)}"

)

if len(bar_failed) > 0:

    print()

    display(

        pd.DataFrame(bar_failed)

    )

print()

print_success(

    "Bar Plot seluruh sampel berhasil dibuat."

)

GENERATE BAR PLOT


Generate Bar Plot:   0%|          | 0/30 [00:00<?, ?it/s]


ℹ️ Berhasil : 30
ℹ️ Gagal : 0

✅ Bar Plot seluruh sampel berhasil dibuat.


## 6.17 Validasi Bar Plot

In [ ]:
# =====================================================
# CELL 50 : VALIDASI BAR PLOT
# =====================================================

print_header("VALIDASI BAR PLOT")

bar_total = count_files(

    LOCAL_BAR_DIR,

    ".png"

)

print()

print_info(

    f"Bar Plot : {bar_total}"

)

print_info(

    f"Expected : {len(SHAP_RESULTS)}"

)

print()

if bar_total == len(SHAP_RESULTS):

    print_success(

        "Seluruh Bar Plot berhasil dibuat."

    )

else:

    print_warning(

        f"Masih terdapat {len(SHAP_RESULTS)-bar_total} file yang belum dibuat."

    )

VALIDASI BAR PLOT

ℹ️ Bar Plot : 30
ℹ️ Expected : 30

✅ Seluruh Bar Plot berhasil dibuat.


## 6.18 Generate Metadata SHAP Local

In [ ]:
# =====================================================
# CELL 51 : GENERATE METADATA SHAP LOCAL
# =====================================================

print_header("GENERATE METADATA SHAP LOCAL")

metadata_success = 0

for sample_id, sample_data in tqdm(

    SHAP_RESULTS.items(),

    desc="Generate Metadata"

):

    metadata = {

        "sample_id":

            sample_data["sample_id"],

        "text":

            sample_data["text"],

        "actual_label":

            sample_data["actual_label"],

        "actual_sentiment":

            sample_data["actual_sentiment"],

        "predicted_label":

            sample_data["predicted_label"],

        "predicted_sentiment":

            sample_data["predicted_sentiment"],

        "confidence":

            float(sample_data["confidence"]),

        "sample_group":

            sample_data["sample_group"],

        "prediction_case":

            sample_data["prediction_case"],

        "confidence_level":

            sample_data["confidence_level"],

        "selection_reason":

            sample_data["selection_reason"],

        "statistics":

            sample_data["statistics"],

        "plots":

            sample_data["plots"]

    }

    output_path = os.path.join(

        LOCAL_METADATA_DIR,

        f"{sample_id}_metadata.json"

    )

    with open(

        output_path,

        "w",

        encoding="utf-8"

    ) as f:

        json.dump(

            metadata,

            f,

            indent=4,

            ensure_ascii=False

        )

    metadata_success += 1

print()

print_info(

    f"Metadata dibuat : {metadata_success}"

)

print()

print_success(

    "Metadata SHAP Local berhasil dibuat."

)

GENERATE METADATA SHAP LOCAL


Generate Metadata:   0%|          | 0/30 [00:00<?, ?it/s]


ℹ️ Metadata dibuat : 30

✅ Metadata SHAP Local berhasil dibuat.


## 6.19 Validasi Metadata SHAP Local

In [ ]:
# =====================================================
# CELL 52 : VALIDASI METADATA SHAP LOCAL
# =====================================================

print_header("VALIDASI METADATA SHAP LOCAL")

metadata_total = count_files(

    LOCAL_METADATA_DIR,

    ".json"

)

print()

print_info(

    f"Metadata : {metadata_total}"

)

print_info(

    f"Expected : {len(SHAP_RESULTS)}"

)

print()

if metadata_total == len(SHAP_RESULTS):

    print_success(

        "Seluruh Metadata berhasil dibuat."

    )

else:

    print_warning(

        f"Masih terdapat {len(SHAP_RESULTS)-metadata_total} metadata yang belum dibuat."

    )

VALIDASI METADATA SHAP LOCAL

ℹ️ Metadata : 30
ℹ️ Expected : 30

✅ Seluruh Metadata berhasil dibuat.


## 6.20 Generate Summary SHAP Local

In [ ]:
# =====================================================
# CELL 53 : GENERATE SUMMARY SHAP LOCAL
# =====================================================

print_header("GENERATE SUMMARY SHAP LOCAL")

summary_rows = []

for sample_id, sample_data in SHAP_RESULTS.items():

    stat = sample_data["statistics"]

    summary_rows.append({

        "sample_id":

            sample_id,

        "actual_sentiment":

            sample_data["actual_sentiment"],

        "predicted_sentiment":

            sample_data["predicted_sentiment"],

        "confidence":

            sample_data["confidence"],

        "sample_group":

            sample_data["sample_group"],

        "top_token":

            stat["top_token"],

        "top_positive_token":

            stat["top_positive_token"],

        "top_negative_token":

            stat["top_negative_token"],

        "mean_abs_shap":

            stat["mean_abs_shap"],

        "max_abs_shap":

            stat["max_abs_shap"]

    })

summary_df = pd.DataFrame(summary_rows)

summary_path = os.path.join(

    CSV_DIR,

    "summary_local_shap.csv"

)

summary_df.to_csv(

    summary_path,

    index=False,

    encoding="utf-8-sig"

)

print()

print_info(

    f"Jumlah Baris : {len(summary_df)}"

)

print_info(

    f"Lokasi : {summary_path}"

)

print()

print_success(

    "Summary SHAP Local berhasil dibuat."

)

GENERATE SUMMARY SHAP LOCAL

ℹ️ Jumlah Baris : 30
ℹ️ Lokasi : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/csv/summary_local_shap.csv

✅ Summary SHAP Local berhasil dibuat.


## 6.21 Validasi Summary SHAP Local

In [ ]:
# =====================================================
# CELL 54 : VALIDASI SUMMARY SHAP LOCAL
# =====================================================

print_header("VALIDASI SUMMARY SHAP LOCAL")

summary_df = pd.read_csv(

    summary_path

)

print()

print_info(

    f"Jumlah Baris : {summary_df.shape[0]}"

)

print_info(

    f"Jumlah Kolom : {summary_df.shape[1]}"

)

print()

if len(summary_df) == len(SHAP_RESULTS):

    print_success(

        "Summary CSV berhasil divalidasi."

    )

else:

    print_warning(

        "Jumlah baris Summary CSV tidak sesuai."

    )

VALIDASI SUMMARY SHAP LOCAL

ℹ️ Jumlah Baris : 30
ℹ️ Jumlah Kolom : 10

✅ Summary CSV berhasil divalidasi.


## 6.22 Generate README SHAP Local

In [ ]:
# =====================================================
# CELL 55 : GENERATE README SHAP LOCAL
# =====================================================

print_header("GENERATE README SHAP LOCAL")

readme_text = f"""
XC_SHAP_ANALYSIS

Jumlah Sampel
-------------
{len(SHAP_RESULTS)}

Output Folder
-------------
csv/
local/
global/
overall/
log/

Isi Folder Local
----------------
- text_plot
- bar_plot
- waterfall_plot
- metadata

File CSV
--------
summary_local_shap.csv

Notebook
--------
XC_SHAP_Analysis.ipynb

Generated Automatically
"""

readme_path = os.path.join(

    README_DIR,

    "README.txt"

)

with open(

    readme_path,

    "w",

    encoding="utf-8"

) as f:

    f.write(

        readme_text

    )

print()

print_info(

    f"Lokasi : {readme_path}"

)

print()

print_success(

    "README berhasil dibuat."

)

GENERATE README SHAP LOCAL

ℹ️ Lokasi : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/readme/README.txt

✅ README berhasil dibuat.


## 6.23 Validasi Akhir SHAP Local

In [ ]:
# =====================================================
# CELL 56 : VALIDASI AKHIR SHAP LOCAL
# =====================================================

print_header("VALIDASI AKHIR SHAP LOCAL")

validation = {

    "Waterfall Plot":

        count_files(

            LOCAL_WATERFALL_DIR,

            ".png"

        ),

    "Bar Plot":

        count_files(

            LOCAL_BAR_DIR,

            ".png"

        ),

    "Metadata":

        count_files(

            LOCAL_METADATA_DIR,

            ".json"

        ),

    "Summary CSV":

        count_files(

            CSV_DIR,

            ".csv"

        ),

    "README":

        count_files(

            README_DIR,

            ".txt"

        )

}

validation_df = pd.DataFrame({

    "Komponen":

        validation.keys(),

    "Jumlah":

        validation.values()

})

display(validation_df)

print()

expected = len(SHAP_RESULTS)

success = True

if validation["Waterfall Plot"] != expected:

    success = False

if validation["Bar Plot"] != expected:

    success = False

if validation["Metadata"] != expected:

    success = False

if validation["Summary CSV"] < 1:

    success = False

if validation["README"] < 1:

    success = False

print_info(

    f"Jumlah Sampel : {expected}"

)

print()

if success:

    print_success(

        "Seluruh output SHAP Local berhasil dibuat."

    )

else:

    print_warning(

        "Masih terdapat output yang belum lengkap."

    )

VALIDASI AKHIR SHAP LOCAL


,Komponen,Jumlah
0,Waterfall Plot,30
1,Bar Plot,30
2,Metadata,30
3,Summary CSV,3
4,README,3



ℹ️ Jumlah Sampel : 30

✅ Seluruh output SHAP Local berhasil dibuat.


# 7. SHAP Global Analysis

## 7.1 Mempersiapkan Global SHAP Values

In [ ]:
# =====================================================
# CELL 57 : PERSIAPAN GLOBAL TOKEN STATISTICS
# =====================================================

print_header("PERSIAPAN GLOBAL TOKEN STATISTICS")

global_token_tables = []

processed_sample_ids = []

for sample_id, sample_data in SHAP_RESULTS.items():

    global_token_tables.append(

        sample_data["token_statistics"].copy()

    )

    processed_sample_ids.append(sample_id)

print()

print_info(

    f"Jumlah Token Statistics : {len(global_token_tables)}"

)

print_info(

    f"Jumlah Sampel : {len(processed_sample_ids)}"

)

print()

print_success(

    "Global Token Statistics berhasil dipersiapkan."

)

PERSIAPAN GLOBAL TOKEN STATISTICS

ℹ️ Jumlah Token Statistics : 30
ℹ️ Jumlah Sampel : 30

✅ Global Token Statistics berhasil dipersiapkan.


## 7.2 Validasi Global SHAP Values

In [ ]:
# =====================================================
# CELL 58 : VALIDASI GLOBAL TOKEN STATISTICS
# =====================================================

print_header("VALIDASI GLOBAL TOKEN STATISTICS")

expected = len(SHAP_RESULTS)

obtained = len(global_token_tables)

print()

print_info(f"Expected : {expected}")

print_info(f"Obtained : {obtained}")

print()

if expected == obtained:

    print_success(

        "Seluruh Token Statistics berhasil dikumpulkan."

    )

else:

    print_warning(

        "Jumlah Token Statistics tidak sesuai."

    )

VALIDASI GLOBAL TOKEN STATISTICS

ℹ️ Expected : 30
ℹ️ Obtained : 30

✅ Seluruh Token Statistics berhasil dikumpulkan.


## 7.3 Generate Global Summary Plot

In [ ]:
# =====================================================
# CELL 59 : GENERATE GLOBAL TOKEN IMPORTANCE
# =====================================================

print_header("GENERATE GLOBAL TOKEN IMPORTANCE")

from collections import defaultdict

token_statistics = defaultdict(

    lambda:{

        "sum_abs_shap":0.0,

        "sum_shap":0.0,

        "frequency":0

    }

)

for token_df in global_token_tables:

    for _, row in token_df.iterrows():

        token = str(row["Token"]).strip()

        if token == "":

            continue

        token_statistics[token]["sum_abs_shap"] += float(

            row["|SHAP|"]

        )

        token_statistics[token]["sum_shap"] += float(

            row["SHAP Value"]

        )

        token_statistics[token]["frequency"] += 1

global_token_df = pd.DataFrame(

    token_statistics

).T.reset_index()

global_token_df.columns = [

    "Token",

    "sum_abs_shap",

    "sum_shap",

    "frequency"

]

global_token_df["mean_abs_shap"] = (

    global_token_df["sum_abs_shap"]

    /

    global_token_df["frequency"]

)

global_token_df["mean_shap"] = (

    global_token_df["sum_shap"]

    /

    global_token_df["frequency"]

)

global_token_df = global_token_df.sort_values(

    "mean_abs_shap",

    ascending=False

).reset_index(drop=True)

display(

    global_token_df.head(20)

)

print()

print_success(

    "Global Token Importance berhasil dibuat."

)

GENERATE GLOBAL TOKEN IMPORTANCE


,Token,sum_abs_shap,sum_shap,frequency,mean_abs_shap,mean_shap
0,buruk,8.176838,8.176838,1.0,8.176838,8.176838
1,jelek,8.097133,8.097133,1.0,8.097133,8.097133
2,tolol,8.056546,8.056546,1.0,8.056546,8.056546
3,parah,6.152412,6.152412,1.0,6.152412,6.152412
4,ditingkatkan,5.262022,5.262022,1.0,5.262022,5.262022
5,membantu,22.482087,22.482087,5.0,4.496417,4.496417
6,nyaman,3.933029,3.933029,1.0,3.933029,3.933029
7,mudah,6.941836,6.941836,2.0,3.470918,3.470918
8,guna,2.929958,-2.929958,1.0,2.929958,-2.929958
9,perjalanan,2.312078,2.312078,1.0,2.312078,2.312078



✅ Global Token Importance berhasil dibuat.


## 7.4 Validasi Global Summary Plot

In [ ]:
# =====================================================
# CELL 60 : VALIDASI GLOBAL TOKEN IMPORTANCE
# =====================================================

print_header("VALIDASI GLOBAL TOKEN IMPORTANCE")

print()

print_info(

    f"Jumlah Token : {len(global_token_df)}"

)

print_info(

    f"Top Token : {global_token_df.iloc[0]['Token']}"

)

print_info(

    f"Mean |SHAP| : {global_token_df.iloc[0]['mean_abs_shap']:.6f}"

)

print()

print_success(

    "Global Token Importance berhasil divalidasi."

)

VALIDASI GLOBAL TOKEN IMPORTANCE

ℹ️ Jumlah Token : 262
ℹ️ Top Token : buruk
ℹ️ Mean |SHAP| : 8.176838

✅ Global Token Importance berhasil divalidasi.


## 7.5 Generate Global Bar Plot

In [ ]:
# =====================================================
# CELL 61 : GENERATE GLOBAL BAR PLOT
# =====================================================

print_header("GENERATE GLOBAL BAR PLOT")

TOP_N = 20

plot_df = global_token_df.head(TOP_N)

global_bar_path = os.path.join(

    GLOBAL_DIR,

    "global_token_importance.png"

)

plt.figure(figsize=(12,8))

plt.barh(

    plot_df["Token"][::-1],

    plot_df["mean_abs_shap"][::-1]

)

plt.xlabel(

    "Mean |SHAP|"

)

plt.ylabel(

    "Token"

)

plt.title(

    "Global Token Importance"

)

plt.tight_layout()

plt.savefig(

    global_bar_path,

    dpi=300,

    bbox_inches="tight"

)

plt.close()

print()

print_info(

    f"Output : {global_bar_path}"

)

print()

print_success(

    "Global Bar Plot berhasil dibuat."

)

GENERATE GLOBAL BAR PLOT

ℹ️ Output : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/global/global_token_importance.png

✅ Global Bar Plot berhasil dibuat.


## 7.6 Validasi Global Bar Plot

In [ ]:
# =====================================================
# CELL 62 : VALIDASI GLOBAL BAR PLOT
# =====================================================

print_header("VALIDASI GLOBAL BAR PLOT")

print()

print_info(

    f"File : {os.path.basename(global_bar_path)}"

)

print_info(

    f"Lokasi : {global_bar_path}"

)

print()

if os.path.exists(global_bar_path):

    print_success(

        "Global Bar Plot berhasil disimpan."

    )

else:

    raise FileNotFoundError(

        global_bar_path

    )

VALIDASI GLOBAL BAR PLOT

ℹ️ File : global_token_importance.png
ℹ️ Lokasi : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/global/global_token_importance.png

✅ Global Bar Plot berhasil disimpan.


## 7.7 Generate Metadata Global SHAP

In [ ]:
# =====================================================
# CELL 63 : GENERATE METADATA GLOBAL SHAP
# =====================================================

print_header("GENERATE METADATA GLOBAL SHAP")

global_metadata = {

    "total_sample": len(SHAP_RESULTS),

    "total_unique_token": int(len(global_token_df)),

    "top_token": global_token_df.iloc[0]["Token"],

    "top_mean_abs_shap": float(

        global_token_df.iloc[0]["mean_abs_shap"]

    ),

    "created_plot": os.path.basename(

        global_bar_path

    ),

    "created_time": pd.Timestamp.now().strftime(

        "%Y-%m-%d %H:%M:%S"

    )

}

global_metadata_path = os.path.join(

    GLOBAL_DIR,

    "global_metadata.json"

)

with open(

    global_metadata_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        global_metadata,

        f,

        indent=4,

        ensure_ascii=False

    )

print()

print_info(

    f"Output : {global_metadata_path}"

)

print()

print_success(

    "Metadata Global SHAP berhasil dibuat."

)

GENERATE METADATA GLOBAL SHAP

ℹ️ Output : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/global/global_metadata.json

✅ Metadata Global SHAP berhasil dibuat.


## 7.8 Validasi Metadata Global SHAP

In [ ]:
# =====================================================
# CELL 64 : VALIDASI METADATA GLOBAL SHAP
# =====================================================

print_header("VALIDASI METADATA GLOBAL SHAP")

print()

print_info(

    f"File : {os.path.basename(global_metadata_path)}"

)

print_info(

    f"Jumlah Token : {global_metadata['total_unique_token']}"

)

print_info(

    f"Top Token : {global_metadata['top_token']}"

)

print()

if os.path.exists(global_metadata_path):

    print_success(

        "Metadata Global berhasil disimpan."

    )

else:

    raise FileNotFoundError(

        global_metadata_path

    )

VALIDASI METADATA GLOBAL SHAP

ℹ️ File : global_metadata.json
ℹ️ Jumlah Token : 262
ℹ️ Top Token : buruk

✅ Metadata Global berhasil disimpan.


## 7.9 Generate Global Summary CSV

In [ ]:
# =====================================================
# CELL 65 : GENERATE GLOBAL SUMMARY CSV
# =====================================================

print_header("GENERATE GLOBAL SUMMARY CSV")

global_summary_csv = os.path.join(

    CSV_DIR,

    "summary_global_shap.csv"

)

global_token_df.to_csv(

    global_summary_csv,

    index=False,

    encoding="utf-8-sig"

)

print()

print_info(

    f"Output : {global_summary_csv}"

)

print()

print_success(

    "Summary Global CSV berhasil disimpan."

)

GENERATE GLOBAL SUMMARY CSV

ℹ️ Output : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/csv/summary_global_shap.csv

✅ Summary Global CSV berhasil disimpan.


## 7.10 Validasi Global Summary CSV

In [ ]:
# =====================================================
# CELL 66 : VALIDASI GLOBAL SUMMARY CSV
# =====================================================

print_header("VALIDASI GLOBAL SUMMARY CSV")

df = pd.read_csv(

    global_summary_csv

)

print()

print_info(

    f"Baris : {len(df)}"

)

print_info(

    f"Kolom : {len(df.columns)}"

)

print()

print_success(

    "Summary Global CSV berhasil divalidasi."

)

VALIDASI GLOBAL SUMMARY CSV

ℹ️ Baris : 262
ℹ️ Kolom : 6

✅ Summary Global CSV berhasil divalidasi.


## 7.11 Generate README Global SHAP

In [ ]:
# =====================================================
# CELL 67 : GENERATE README GLOBAL SHAP
# =====================================================

print_header("GENERATE README GLOBAL SHAP")

readme_path = os.path.join(

    README_DIR,

    "README_GLOBAL_SHAP.txt"

)

with open(

    readme_path,

    "w",

    encoding="utf-8"

) as f:

    f.write(

"""GLOBAL SHAP ANALYSIS

Folder ini berisi:

1. Global Token Importance

2. Global Metadata

3. Global Summary CSV

4. Global Bar Plot

Analisis dilakukan terhadap 30 sampel
yang telah dipilih secara purposive.

"""
    )

print()

print_info(

    f"Output : {readme_path}"

)

print()

print_success(

    "README Global berhasil dibuat."

)

GENERATE README GLOBAL SHAP

ℹ️ Output : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/readme/README_GLOBAL_SHAP.txt

✅ README Global berhasil dibuat.


## 7.12 Validasi README Global SHAP

In [ ]:
# =====================================================
# CELL 68 : VALIDASI README GLOBAL SHAP
# =====================================================

print_header("VALIDASI README GLOBAL SHAP")

print()

print_info(

    f"File : {os.path.basename(readme_path)}"

)

print()

if os.path.exists(readme_path):

    print_success(

        "README Global berhasil disimpan."

    )

else:

    raise FileNotFoundError(

        readme_path

    )

VALIDASI README GLOBAL SHAP

ℹ️ File : README_GLOBAL_SHAP.txt

✅ README Global berhasil disimpan.


# 8. ANALISIS GLOBAL HASIL SHAP

## 8.1 Generate Overall SHAP Analysis

In [ ]:
# =====================================================
# CELL 69 : GENERATE OVERALL SHAP ANALYSIS
# =====================================================

print_header("GENERATE OVERALL SHAP ANALYSIS")

TOP_N = 20

overall_summary = global_token_df.head(TOP_N).copy()

overall_summary["Ranking"] = np.arange(

    1,

    len(overall_summary)+1

)

overall_summary = overall_summary[

    [

        "Ranking",

        "Token",

        "frequency",

        "mean_abs_shap",

        "mean_shap"

    ]

]

display(overall_summary)

print()

print_success(

    "Overall SHAP Analysis berhasil dibuat."

)

GENERATE OVERALL SHAP ANALYSIS


,Ranking,Token,frequency,mean_abs_shap,mean_shap
0,1,buruk,1.0,8.176838,8.176838
1,2,jelek,1.0,8.097133,8.097133
2,3,tolol,1.0,8.056546,8.056546
3,4,parah,1.0,6.152412,6.152412
4,5,ditingkatkan,1.0,5.262022,5.262022
5,6,membantu,5.0,4.496417,4.496417
6,7,nyaman,1.0,3.933029,3.933029
7,8,mudah,2.0,3.470918,3.470918
8,9,guna,1.0,2.929958,-2.929958
9,10,perjalanan,1.0,2.312078,2.312078



✅ Overall SHAP Analysis berhasil dibuat.


## 8.2 Validasi Overall SHAP Analysis

In [ ]:
# =====================================================
# CELL 70 : VALIDASI OVERALL SHAP ANALYSIS
# =====================================================

print_header("VALIDASI OVERALL SHAP ANALYSIS")

print()

print_info(

    f"Jumlah Token : {len(overall_summary)}"

)

print_info(

    f"Token Peringkat 1 : {overall_summary.iloc[0]['Token']}"

)

print_info(

    f"Mean |SHAP| : {overall_summary.iloc[0]['mean_abs_shap']:.6f}"

)

print()

print_success(

    "Overall SHAP Analysis berhasil divalidasi."

)

VALIDASI OVERALL SHAP ANALYSIS

ℹ️ Jumlah Token : 20
ℹ️ Token Peringkat 1 : buruk
ℹ️ Mean |SHAP| : 8.176838

✅ Overall SHAP Analysis berhasil divalidasi.


## 8.3 Simpan Overall SHAP Analysis

In [ ]:
# =====================================================
# CELL 71 : SIMPAN OVERALL SHAP ANALYSIS
# =====================================================

print_header("SIMPAN OVERALL SHAP ANALYSIS")

overall_csv_path = os.path.join(

    CSV_DIR,

    "overall_shap_analysis.csv"

)

overall_summary.to_csv(

    overall_csv_path,

    index=False,

    encoding="utf-8-sig"

)

print()

print_info(

    f"Output : {overall_csv_path}"

)

print()

print_success(

    "Overall SHAP Analysis berhasil disimpan."

)

SIMPAN OVERALL SHAP ANALYSIS

ℹ️ Output : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/csv/overall_shap_analysis.csv

✅ Overall SHAP Analysis berhasil disimpan.


## 8.4 Validasi Overall SHAP Analysis

In [ ]:
# =====================================================
# CELL 72 : VALIDASI OVERALL SHAP ANALYSIS CSV
# =====================================================

print_header("VALIDASI OVERALL SHAP ANALYSIS CSV")

df = pd.read_csv(

    overall_csv_path

)

print()

print_info(

    f"Jumlah Baris : {len(df)}"

)

print_info(

    f"Jumlah Kolom : {len(df.columns)}"

)

print()

if os.path.exists(overall_csv_path):

    print_success(

        "Overall SHAP CSV berhasil disimpan."

    )

else:

    raise FileNotFoundError(

        overall_csv_path

    )

VALIDASI OVERALL SHAP ANALYSIS CSV

ℹ️ Jumlah Baris : 20
ℹ️ Jumlah Kolom : 5

✅ Overall SHAP CSV berhasil disimpan.


## 8.5 Validasi Seluruh Output XC

In [ ]:
# =====================================================
# CELL 73 : VALIDASI SELURUH OUTPUT XC
# =====================================================

print_header("VALIDASI SELURUH OUTPUT XC")

validation = {

    "Local Waterfall Plot":

        count_files(

            LOCAL_WATERFALL_DIR,

            ".png"

        ),

    "Local Bar Plot":

        count_files(

            LOCAL_BAR_DIR,

            ".png"

        ),

    "Local Metadata":

        count_files(

            LOCAL_METADATA_DIR,

            ".json"

        ),

    "Local CSV":

        count_files(

            CSV_DIR,

            ".csv"

        ),

    "Global Plot":

        count_files(

            GLOBAL_DIR,

            ".png"

        ),

    "Global Metadata":

        count_files(

            GLOBAL_DIR,

            ".json"

        ),

    "README":

        count_files(

            README_DIR,

            ".txt"

        )

}

validation_df = pd.DataFrame({

    "Komponen": validation.keys(),

    "Jumlah": validation.values()

})

display(validation_df)

print()

print_success(

    "Seluruh komponen XC berhasil diperiksa."

)

VALIDASI SELURUH OUTPUT XC


,Komponen,Jumlah
0,Local Waterfall Plot,30
1,Local Bar Plot,30
2,Local Metadata,30
3,Local CSV,3
4,Global Plot,1
5,Global Metadata,1
6,README,3



✅ Seluruh komponen XC berhasil diperiksa.


## 8.6 Final Validation

In [ ]:
# =====================================================
# CELL 74 : FINAL VALIDATION XC
# =====================================================

print_header("FINAL VALIDATION XC")

success = True

required_files = [

    global_bar_path,

    global_metadata_path,

    global_summary_csv,

    overall_csv_path,

    readme_path

]

for path in required_files:

    if not os.path.exists(path):

        success = False

print()

print_info(

    f"Jumlah Sampel : {len(SHAP_RESULTS)}"

)

print_info(

    f"Jumlah Token Global : {len(global_token_df)}"

)

print_info(

    f"Overall Token : {len(overall_summary)}"

)

print()

if success:

    print_success(

        "Notebook XC berhasil diselesaikan."

    )

else:

    print_warning(

        "Masih terdapat output yang belum lengkap."

    )

FINAL VALIDATION XC

ℹ️ Jumlah Sampel : 30
ℹ️ Jumlah Token Global : 262
ℹ️ Overall Token : 20

✅ Notebook XC berhasil diselesaikan.


## 8.7 Ringkasan Notebook XC

In [ ]:
# =====================================================
# CELL 75 : NOTEBOOK XC COMPLETED
# =====================================================

print("="*60)
print("NOTEBOOK XC COMPLETED")
print("="*60)

print()

print("✓ Local SHAP Analysis")
print("✓ Global SHAP Analysis")
print("✓ Overall SHAP Analysis")
print("✓ Metadata")
print("✓ CSV")
print("✓ README")
print("✓ Final Validation")

print()

print_info(
    f"Total Sampel : {len(SHAP_RESULTS)}"
)

print_info(
    f"Total Token : {len(global_token_df)}"
)

print_info(
    f"Output Folder : {GLOBAL_DIR}"
)

print()

print_success(
    "Notebook XC berhasil diselesaikan 100%."
)

NOTEBOOK XC COMPLETED

✓ Local SHAP Analysis
✓ Global SHAP Analysis
✓ Overall SHAP Analysis
✓ Metadata
✓ CSV
✓ README
✓ Final Validation

ℹ️ Total Sampel : 30
ℹ️ Total Token : 262
ℹ️ Output Folder : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/global

✅ Notebook XC berhasil diselesaikan 100%.


## 8.7 Generate Output Manifest

In [ ]:
# =====================================================
# CELL 75 : GENERATE OUTPUT MANIFEST
# =====================================================

print_header("GENERATE OUTPUT MANIFEST")

# Root folder XC otomatis
ROOT_DIR = os.path.dirname(GLOBAL_DIR)

manifest = {

    "project": "XC_SHAP_Analysis",

    "total_sample": len(SHAP_RESULTS),

    "total_global_token": len(global_token_df),

    "total_overall_token": len(overall_summary),

    "local_waterfall_plot":

        count_files(

            LOCAL_WATERFALL_DIR,

            ".png"

        ),

    "local_bar_plot":

        count_files(

            LOCAL_BAR_DIR,

            ".png"

        ),

    "local_metadata":

        count_files(

            LOCAL_METADATA_DIR,

            ".json"

        ),

    "csv":

        count_files(

            CSV_DIR,

            ".csv"

        ),

    "global_plot":

        count_files(

            GLOBAL_DIR,

            ".png"

        ),

    "readme":

        count_files(

            README_DIR,

            ".txt"

        )

}

manifest_path = os.path.join(

    ROOT_DIR,

    "manifest_xc.json"

)

with open(

    manifest_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        manifest,

        f,

        indent=4,

        ensure_ascii=False

    )

print()

print_info(

    f"Manifest : {manifest_path}"

)

print()

print_success(

    "Output Manifest berhasil dibuat."

)

GENERATE OUTPUT MANIFEST

ℹ️ Manifest : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/manifest_xc.json

✅ Output Manifest berhasil dibuat.


## 8.8 Notebook XC Completed

In [ ]:
# =====================================================
# CELL 76 : NOTEBOOK XC COMPLETED
# =====================================================

print("="*60)
print("NOTEBOOK XC COMPLETED")
print("="*60)

print()

summary = pd.DataFrame({

    "Komponen":[

        "Local Waterfall",

        "Local Bar",

        "Metadata",

        "CSV",

        "Global Plot",

        "README"

    ],

    "Jumlah":[

        count_files(

            LOCAL_WATERFALL_DIR,

            ".png"

        ),

        count_files(

            LOCAL_BAR_DIR,

            ".png"

        ),

        count_files(

            LOCAL_METADATA_DIR,

            ".json"

        ),

        count_files(

            CSV_DIR,

            ".csv"

        ),

        count_files(

            GLOBAL_DIR,

            ".png"

        ),

        count_files(

            README_DIR,

            ".txt"

        )

    ]

})

display(summary)

print()

print_info(

    f"Total Sampel : {len(SHAP_RESULTS)}"

)

print_info(

    f"Total Token Global : {len(global_token_df)}"

)

print_info(

    f"Manifest : {manifest_path}"

)

print()

print_success(

    "Notebook XC berhasil diselesaikan 100%."

)

NOTEBOOK XC COMPLETED



,Komponen,Jumlah
0,Local Waterfall,30
1,Local Bar,30
2,Metadata,30
3,CSV,3
4,Global Plot,1
5,README,3



ℹ️ Total Sampel : 30
ℹ️ Total Token Global : 262
ℹ️ Manifest : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis/manifest_xc.json

✅ Notebook XC berhasil diselesaikan 100%.
